# Evaluasi Metode Penanganan Missing Value pada Air-Quality Dataset

**Tujuan**

Menentukan metode imputasi yang layak digunakan untuk setiap variabel dan kategori panjang gap berdasarkan hasil pengujian terhadap nilai asli yang diketahui (*ground truth*). Evaluasi tidak hanya bertujuan memilih metode dengan rata-rata RMSE terendah, tetapi juga menilai konsistensi performa metode pada trial konfirmasi, pengaruh kondisi *co-missing*, serta ketersediaan gap aktual sebagai dasar kekuatan dukungan data.

Variabel yang dievaluasi meliputi `PM2.5`, `PM10`, `DEWP`, `TEMP`, `PRES`, `RAIN`, dan `WSPM`.

**Ide**

- **Ground Truth:** Menggunakan baris *complete-case*, yaitu baris yang memiliki nilai lengkap pada variabel target dan seluruh fitur meteorologi yang digunakan dalam evaluasi. Nilai aslinya disimpan sebagai acuan untuk mengukur kesalahan hasil imputasi.
- **Gap Buatan (*Artificial Missingness*):** Membentuk blok *missing value* pada nilai target dengan menyembunyikan sejumlah nilai dalam rentang jam berurutan. Panjang gap diambil berdasarkan distribusi gap aktual pada data asli, bukan ditentukan secara sembarang.
- **Dua Skenario Simulasi:**
  - `independent`: simulasi gap pada satu stasiun tanpa menambahkan pola *missing value* dari stasiun lain.
  - `co-missing`: simulasi gap pada satu stasiun dengan menambahkan pola kehilangan data secara bersamaan pada stasiun lain berdasarkan template gap aktual.
- **Enam Metode Imputasi:** Seluruh metode diuji pada gap buatan yang sama, yaitu:
  1. Linear Interpolation
  2. Conditional Median
  3. KNN Imputer
  4. Neighbor Station
  5. Interp + NC (level)
  6. Interp + NC (anomaly)
- **Evaluasi Performa:** Membandingkan hasil imputasi dengan nilai *ground truth* menggunakan metrik RMSE, MAE, bias, dan rasio standar deviasi. Evaluasi juga mempertimbangkan jumlah titik yang berhasil dievaluasi serta *fallback rate* apabila metode menggunakan mekanisme pengganti.
- **Seleksi dan Konfirmasi Metode:** Membagi trial berdasarkan random seed menjadi dua fase:
  - **Fase seleksi:** Menentukan metode terbaik berdasarkan rata-rata RMSE dan mengidentifikasi metode peringkat kedua.
  - **Fase konfirmasi:** Menguji kembali metode terpilih menggunakan seed berbeda dan membandingkan performanya terhadap Linear Interpolation sebagai metode referensi.
- **Analisis Dampak Co-Missing:** Membandingkan perubahan RMSE antara skenario `independent` dan `co-missing` untuk mengetahui bagaimana kehilangan data secara bersamaan pada stasiun lain berkaitan dengan perubahan performa metode imputasi.
- **Diagnostik Co-Missing Aktual dan Simulasi:** Membandingkan proporsi jam ketika minimal satu atau seluruh stasiun lain mengalami *missing value* pada gap aktual dan simulasi. Tahapan ini digunakan untuk menilai kesesuaian pola *co-missing* yang dibentuk dalam simulasi terhadap kondisi data asli.
- **Penyusunan Aturan Penanganan Missing Value:** Menyusun rekomendasi berdasarkan hasil konfirmasi pada skenario terakhir, dengan memisahkan dua informasi:
  - **Status keputusan metode:** `didukung`, `batas`, atau `tidak ada gap aktual`.
  - **Kekuatan dukungan data:** `memadai`, `terbatas`, atau `lemah`.

  Kedua informasi tersebut digunakan bersama untuk menentukan tingkat keyakinan terhadap rekomendasi metode imputasi.

Prasyarat

- Sel 3 membaca 12 berkas `PRSA_Data_*.csv` dari direktori `data_directory` dan menggabungkannya menjadi `air_df`.
- Path `data_directory` perlu disesuaikan dengan lokasi dataset pada komputer yang digunakan.
- Dataset harus memiliki kolom `year`, `month`, `day`, `hour`, `station`, serta tujuh variabel target yang dievaluasi.
- Jika `air_df` sudah tersedia di memori dari notebook lain, sel 3 dapat dilewati.
- Notebook dijalankan tanpa menulis atau mengubah berkas dataset asli.

**Batasan Umum yang Berlaku untuk Seluruh Hasil**
- **Keterbatasan Complete-Case**
  - Evaluasi hanya menggunakan baris *complete-case*, sehingga hasilnya tidak sepenuhnya mewakili kondisi data asli.
  - Baris yang memiliki satu saja fitur meteorologi kosong ikut dikeluarkan, termasuk pada evaluasi metode yang sebenarnya tidak memerlukan seluruh fitur tersebut.

- **Keterbatasan Simulasi Co-Missing**
  - Skenario *co-missing* dibentuk menggunakan template gap aktual, bukan mereplikasi seluruh karakteristik pola *missingness* pada data asli.
  - Stasiun yang tersingkir karena *complete-case* juga dapat memengaruhi cakupan stasiun dalam simulasi.

- **Lokasi Gap Buatan Bersifat Acak**
  - Lokasi waktu gap buatan dipilih secara acak berdasarkan gap aktual, bukan mengikuti waktu kejadian gap asli.
  - Karakteristik musiman, kondisi meteorologi, dan episode polusi pada periode gap aktual tidak dipertahankan secara langsung.

- **Jumlah Blok Simulasi Bukan Jumlah Bukti Aktual**
  - Apabila jumlah gap aktual lebih sedikit daripada blok yang dibutuhkan, gap aktual dapat digunakan kembali dengan pengembalian (*replacement*).
  - Banyaknya blok simulasi tidak dapat dianggap sebagai banyaknya sampel independen.
  - Jumlah gap aktual unik (`available_actual_gaps`), rata-rata gap sumber unik (`unique_source_gaps_mean`), dan penggunaan *replacement* (`replacement_used`) perlu diperhatikan secara terpisah.

- **Keterbatasan Pemisahan Trial**
  - Trial dibagi menjadi fase seleksi dan konfirmasi menggunakan random seed berbeda, dengan konfigurasi default masing-masing lima seed.
  - Pemisahan ini mengurangi bias akibat memilih dan mengevaluasi pemenang pada trial yang sama.
  - Kedua fase masih menggunakan data dasar dan kumpulan gap aktual yang sama, sehingga belum setara dengan validasi pada dataset eksternal atau periode waktu yang benar-benar baru.

- **Keputusan Bersifat Eksploratif**
  - Perbandingan RMSE berpasangan, jumlah kemenangan (*wins*), dan selisih relatif RMSE digunakan sebagai dasar pengambilan keputusan eksploratif.
  - Hasilnya bukan uji hipotesis atau bukti signifikansi statistik, terutama karena jumlah trial dan gap aktual terbatas.

- **Dukungan Data dan Status Keputusan Merupakan Aspek Berbeda**
  - Kategori dukungan data (`memadai`, `terbatas`, dan `lemah`) dilaporkan secara terpisah dari status keputusan metode (`didukung` dan `batas`).
  - Metode yang memperoleh status `didukung` belum tentu memiliki dukungan data yang memadai.
  - Keduanya harus dipertimbangkan ketika membaca tabel aturan akhir.

- **Keterbatasan Interpretasi Co-Missing**
  - Perubahan RMSE antara skenario `independent` dan `co-missing` menunjukkan perbedaan performa pada kondisi simulasi yang diuji, tetapi tidak secara otomatis membuktikan hubungan sebab-akibat.
  - Diagnostik proporsi *co-missing* juga tidak menjamin kesamaan pola waktu, durasi, dan struktur kehilangan data aktual.

- **Keterbatasan Generalisasi Antarvariabel**
  - Hasil evaluasi `PM2.5` dan `PM10` tidak dapat langsung digeneralisasikan ke variabel meteorologi.
  - Variabel meteorologi memiliki jumlah gap aktual yang sangat terbatas, terutama pada kategori gap panjang.
  - Rekomendasi untuk variabel tersebut perlu dibaca dengan mempertimbangkan tingkat dukungan datanya.

- **Keterbatasan Penerapan pada Data Asli**
  - Hasil evaluasi diperoleh dari simulasi pada nilai yang diketahui (*ground truth*).
  - Performa metode ketika diterapkan pada gap asli yang nilai sebenarnya tidak diketahui belum dapat diverifikasi secara langsung melalui eksperimen ini.

**Catatan**

- Hasil eksperimen digunakan sebagai dasar empiris untuk menyusun aturan penanganan *missing value* berdasarkan variabel dan panjang gap.
- Pemilihan metode mempertimbangkan:
  - Konsistensi performa pada fase konfirmasi.
  - Perbandingan terhadap metode referensi.
  - Kondisi *co-missing*.
  - Ketersediaan gap aktual.
- Rekomendasi akhir merupakan aturan penanganan berbasis hasil evaluasi eksploratif.

## Impor pustaka

Menyiapkan pustaka yang dipakai (pathlib, numpy, pandas, scipy, scikit-learn).

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

from scipy import stats
from sklearn.preprocessing import StandardScaler
from sklearn.impute import KNNImputer
from sklearn.metrics import mean_squared_error, mean_absolute_error

## Memuat data

Membuat data frame `air_df` dari 12 berkas stasiun, dengan membaca semua `PRSA_Data_*.csv` di `data_directory`, menggabungkannya, dan menampilkan ringkasan per stasiun. `data_directory` adalah path lokal dan harus disesuaikan.

In [ ]:
data_directory = Path("data")
file_paths = sorted(data_directory.glob("PRSA_Data_*.csv"))

if not file_paths:
    raise FileNotFoundError("Dataset tidak ditemukan di folder 'data'.")

station_dataframes = [pd.read_csv(file_path) for file_path in file_paths]
air_df = pd.concat(station_dataframes, ignore_index=True)

station_summary_df = pd.DataFrame([
    {
        "nama_file": file_path.name,
        "nama_stasiun": dataframe["station"].iloc[0],
        "jumlah_observasi": len(dataframe)
    }
    for file_path, dataframe in zip(file_paths, station_dataframes)
])

print(f"Jumlah berkas stasiun: {len(file_paths)}")
print(f"Ukuran data gabungan: {air_df.shape[0]:,} baris x {air_df.shape[1]} kolom")

display(station_summary_df)
display(air_df.head())

Jumlah berkas stasiun: 12
Ukuran data gabungan: 420,768 baris x 18 kolom


,nama_file,nama_stasiun,jumlah_observasi
0,PRSA_Data_Aotizhongxin_20130301-20170228.csv,Aotizhongxin,35064
1,PRSA_Data_Changping_20130301-20170228.csv,Changping,35064
2,PRSA_Data_Dingling_20130301-20170228.csv,Dingling,35064
3,PRSA_Data_Dongsi_20130301-20170228.csv,Dongsi,35064
4,PRSA_Data_Guanyuan_20130301-20170228.csv,Guanyuan,35064
5,PRSA_Data_Gucheng_20130301-20170228.csv,Gucheng,35064
6,PRSA_Data_Huairou_20130301-20170228.csv,Huairou,35064
7,PRSA_Data_Nongzhanguan_20130301-20170228.csv,Nongzhanguan,35064
8,PRSA_Data_Shunyi_20130301-20170228.csv,Shunyi,35064
9,PRSA_Data_Tiantan_20130301-20170228.csv,Tiantan,35064


,No,year,month,day,hour,PM2.5,PM10,SO2,NO2,CO,O3,TEMP,PRES,DEWP,RAIN,wd,WSPM,station
0,1,2013,3,1,0,4.0,4.0,4.0,7.0,300.0,77.0,-0.7,1023.0,-18.8,0.0,NNW,4.4,Aotizhongxin
1,2,2013,3,1,1,8.0,8.0,4.0,7.0,300.0,77.0,-1.1,1023.2,-18.2,0.0,N,4.7,Aotizhongxin
2,3,2013,3,1,2,7.0,7.0,5.0,10.0,300.0,73.0,-1.1,1023.5,-18.2,0.0,NNW,5.6,Aotizhongxin
3,4,2013,3,1,3,6.0,6.0,11.0,11.0,300.0,72.0,-1.4,1024.5,-19.4,0.0,NW,3.1,Aotizhongxin
4,5,2013,3,1,4,3.0,3.0,12.0,12.0,300.0,72.0,-2.0,1025.2,-19.5,0.0,N,2.0,Aotizhongxin


## Konfigurasi Simulasi dan Evaluasi Imputasi

Tujuan:

Menentukan konfigurasi eksperimen untuk membandingkan beberapa metode imputasi missing values berdasarkan variabel target, panjang gap, pola kehilangan data, dan hasil evaluasi. Konfigurasi ini juga memisahkan data simulasi untuk pemilihan metode dan pengujian konfirmasi agar mengurangi risiko selection bias.

Yang dilakukan:

- Menentukan variabel target (`target_columns`) yang akan diuji, yaitu `PM2.5`, `PM10`, `DEWP`, `TEMP`, `PRES`, `RAIN`, dan `WSPM`.
- Menentukan variabel meteorologi (`meteorological_features`) yang digunakan sebagai fitur pendukung dalam metode imputasi.
- Membagi random seed menjadi dua kelompok, yaitu `selection_seeds` untuk memilih metode terbaik dan `confirmation_seeds` untuk menguji metode terpilih secara terpisah.
- Menentukan skenario simulasi (`scenarios`), yaitu:
  - `independent`: simulasi gap hanya dilakukan pada satu stasiun.
  - `co-missing`: simulasi gap mempertimbangkan pola kehilangan data yang terjadi bersamaan pada stasiun lain.
- Menentukan kategori panjang gap (`gap_scenarios`), mulai dari 1–3 jam hingga lebih dari 72 jam.
- Membatasi jumlah blok gap yang disimulasikan pada setiap kategori melalui `max_blocks_per_gap`.
- Menentukan ukuran data pelatihan dan batch untuk metode KNN melalui `knn_train_size` dan `knn_batch_size`.
- Menentukan variabel yang tidak boleh memiliki nilai negatif (`nonneg_targets`) agar hasil imputasi dapat disesuaikan dengan batasan nilai yang berlaku.
- Menentukan kelompok metode yang memanfaatkan informasi dari stasiun lain (`neighbor_based`).
- Menetapkan interpolasi linear (`reference_method`) sebagai metode pembanding praktis.
- Menentukan ambang peningkatan RMSE minimum (`min_relative_gain`) sebesar 2% untuk menilai apakah keunggulan metode terbaik cukup berarti secara praktis.
- Menentukan jumlah minimum trial konfirmasi (`min_trials_consistent`) sebanyak 3 agar hasil dapat diberi label konsisten.
- Menetapkan batas dukungan data (`weak_support_below` dan `adequate_support_from`) berdasarkan jumlah gap aktual unik untuk menilai kecukupan data pada setiap variabel dan kategori gap.

Batasan:

- Pengujian hanya mencakup tujuh variabel target yang telah ditentukan.
- Variabel meteorologi yang digunakan sebagai fitur pendukung terbatas pada `TEMP`, `PRES`, `DEWP`, `RAIN`, dan `WSPM`.
- Simulasi menggunakan 10 seed yang dibagi menjadi dua kelompok tanpa irisan, masing-masing lima seed.
- Jumlah blok gap dibatasi berdasarkan kategori panjang gap sehingga tidak semua gap aktual dapat digunakan dalam simulasi.
- Evaluasi skenario `independent` dan `co-missing` bergantung pada ketersediaan pola gap aktual yang sesuai.
- Nilai hasil imputasi untuk `PM2.5`, `PM10`, `RAIN`, dan `WSPM` dibatasi agar tidak negatif. Pembatasan ini merupakan aturan pascapemrosesan, bukan jaminan bahwa prediksi model selalu akurat.
- Keunggulan metode dinilai menggunakan RMSE dengan ambang peningkatan relatif minimum 2%, sehingga perbedaan yang lebih kecil belum tentu dianggap bermakna secara praktis.
- Label konsistensi membutuhkan minimal tiga trial konfirmasi, sedangkan kecukupan dukungan data ditentukan berdasarkan jumlah gap aktual unik, bukan jumlah baris hasil simulasi.

In [3]:
target_columns = ["PM2.5", "PM10", "DEWP", "TEMP", "PRES", "RAIN", "WSPM"]
meteorological_features = ["TEMP", "PRES", "DEWP", "RAIN", "WSPM"]

selection_seeds = [42, 123, 202, 456, 789]
confirmation_seeds = [1011, 2027, 3141, 5926, 8675]
random_seeds = selection_seeds + confirmation_seeds

scenarios = ["independent", "co-missing"]

gap_scenarios = {
    "1-3 jam": (1, 3),
    "4-6 jam": (4, 6),
    "7-12 jam": (7, 12),
    "13-24 jam": (13, 24),
    "25-72 jam": (25, 72),
    ">72 jam": (73, np.inf),
}

max_blocks_per_gap = {
    "1-3 jam": 30,
    "4-6 jam": 30,
    "7-12 jam": 20,
    "13-24 jam": 15,
    "25-72 jam": 6,
    ">72 jam": 3,
}

knn_train_size = 3000
knn_batch_size = 1000

nonneg_targets = {"PM2.5", "PM10", "RAIN", "WSPM"}

neighbor_based = {
    "Neighbor Station",
    "Interp + NC (level)",
    "Interp + NC (anomaly)",
}

reference_method = "Linear Interpolation"

min_relative_gain = 0.02

min_trials_consistent = 3

if len(set(random_seeds)) != len(random_seeds):
    raise ValueError("selection_seeds dan confirmation_seeds tidak boleh beririsan atau berulang.")
if len(confirmation_seeds) < min_trials_consistent:
    raise ValueError(
        f"Butuh minimal {min_trials_consistent} seed konfirmasi untuk menilai konsistensi."
    )

weak_support_below = 5
adequate_support_from = 30

## Persiapan Data Ground Truth

Tujuan:

Mempersiapkan data acuan (ground truth) yang memiliki nilai lengkap dan valid untuk digunakan dalam simulasi serta evaluasi kinerja metode imputasi. Data ini menjadi pembanding karena nilai aslinya diketahui sebelum sebagian nilai dibuat hilang secara artifisial.

Yang dilakukan:

- Membuat salinan dataset `air_df` menggunakan `copy()` ke dalam `ground_truth_data` agar proses persiapan tidak mengubah dataset asli.
- Menggabungkan kolom `year`, `month`, `day`, dan `hour` menjadi satu kolom `datetime` menggunakan `pd.to_datetime()` untuk membentuk informasi waktu pengamatan.
- Menghapus baris yang gagal dikonversi menjadi tanggal dan waktu valid menggunakan `dropna()`.
- Mengurutkan data berdasarkan `station` dan `datetime` agar urutan pengamatan setiap stasiun tersusun secara kronologis.
- Memeriksa duplikasi berdasarkan kombinasi `station` dan `datetime`. Jika ditemukan duplikasi, proses dihentikan menggunakan `ValueError`.
- Membuat dictionary `ground_truth` untuk menyimpan data acuan setiap variabel target secara terpisah.
- Menentukan kolom yang wajib tersedia untuk setiap target, meliputi variabel target, identitas stasiun, informasi bulan dan jam, serta seluruh fitur meteorologi.
- Menggunakan `dropna()` untuk mengambil baris complete-case, yaitu baris yang tidak memiliki nilai kosong pada seluruh kolom yang dibutuhkan.
- Mengurutkan kembali data berdasarkan stasiun dan waktu, kemudian menyimpannya ke dalam dictionary berdasarkan nama variabel target.
- Menampilkan jumlah baris data lengkap yang tersedia untuk setiap variabel target sebagai informasi awal ketersediaan data evaluasi.

Batasan:

- Data ground truth hanya mencakup baris yang memiliki nilai lengkap pada variabel target dan seluruh fitur meteorologi. Akibatnya, sebagian data asli tidak digunakan dalam evaluasi.
- Jumlah data ground truth dapat berbeda untuk setiap variabel target karena kelengkapan nilai pada masing-masing variabel berbeda.
- Evaluasi hanya mengukur kinerja imputasi pada kondisi data yang awalnya lengkap, sehingga hasilnya belum tentu sepenuhnya mewakili kondisi ketika missing values terjadi pada data asli.
- Penggunaan complete-case dapat menimbulkan bias apabila pola kehilangan data berkaitan dengan karakteristik pengamatan tertentu.
- Data yang digunakan sebagai ground truth merupakan data observasi, bukan nilai yang telah diverifikasi secara independen sebagai nilai sebenarnya. Oleh karena itu, kesalahan atau anomali yang masih terdapat dalam data asli dapat ikut menjadi bagian dari data acuan.

In [4]:
ground_truth_data = air_df.copy()

ground_truth_data["datetime"] = pd.to_datetime(
    ground_truth_data[["year", "month", "day", "hour"]],
    errors="coerce"
)

ground_truth_data = (
    ground_truth_data
    .dropna(subset=["datetime"])
    .sort_values(["station", "datetime"])
    .reset_index(drop=True)
)

if ground_truth_data.duplicated(["station", "datetime"]).any():
    raise ValueError("Terdapat pasangan (station, datetime) duplikat.")

ground_truth = {}

for target in target_columns:

    required_columns = list(dict.fromkeys(
        [target, "station", "month", "hour"] + meteorological_features
    ))

    complete_data = (
        ground_truth_data
        .dropna(subset=required_columns)
        .sort_values(["station", "datetime"])
        .reset_index(drop=True)
    )

    ground_truth[target] = complete_data

    print(f"{target}: {len(complete_data):,} baris")

PM2.5: 411,621 baris
PM10: 413,913 baris
DEWP: 420,359 baris
TEMP: 420,359 baris
PRES: 420,359 baris
RAIN: 420,359 baris
WSPM: 420,359 baris


Output menunjukkan jumlah baris data yang berhasil dipertahankan sebagai ground truth untuk setiap variabel target setelah dilakukan konversi waktu, pengurutan data berdasarkan stasiun dan waktu, serta penghapusan baris yang memiliki missing value pada salah satu kolom yang dibutuhkan. Jumlah data berbeda karena setiap variabel memiliki tingkat missing value yang berbeda. `PM2.5` memiliki 411.621 baris dan `PM10` sebanyak 413.913 baris, sedangkan variabel meteorologi (`DEWP`, `TEMP`, `PRES`, `RAIN`, dan `WSPM`) masing-masing memiliki 420.359 baris. Perbedaan ini menunjukkan bahwa data `PM2.5` dan `PM10` lebih banyak berkurang karena missing value pada variabel target maupun fitur pendukung. Dengan demikian, setiap variabel memiliki dataset ground truth tersendiri yang hanya berisi baris dengan nilai lengkap pada seluruh kolom yang diperlukan untuk simulasi dan evaluasi metode imputasi.

## Identifikasi Gap Aktual

Tujuan:

Mengidentifikasi dan memetakan pola missing values yang benar-benar terdapat pada dataset berdasarkan urutan waktu pengamatan setiap stasiun. Hasil identifikasi digunakan sebagai dasar untuk menentukan panjang gap dan pola kehilangan data yang akan direpresentasikan dalam simulasi imputasi.

Yang dilakukan:

- Membuat fungsi `identify_actual_gaps()` untuk mengidentifikasi blok missing values yang terjadi secara berurutan pada setiap stasiun dan variabel target.
- Mengambil kolom `station`, `datetime`, dan variabel target yang sedang diperiksa, kemudian mengurutkan data berdasarkan stasiun dan waktu.
- Mengidentifikasi baris yang memiliki nilai kosong menggunakan `isna()`.
- Membentuk segmen pengamatan (`seg_id`) berdasarkan perubahan stasiun atau jeda waktu yang tidak berurutan satu jam. Hal ini dilakukan agar gap tidak dianggap berkesinambungan apabila terdapat jeda waktu dalam pencatatan.
- Membentuk kelompok rangkaian nilai (`run_id`) berdasarkan perubahan status missing dan non-missing untuk mengidentifikasi blok nilai kosong yang berurutan.
- Mengelompokkan data berdasarkan rangkaian tersebut untuk memperoleh informasi stasiun, waktu awal, waktu akhir, panjang gap, status missing, dan segmen pengamatan.
- Memastikan gap memiliki nilai teramati sebelum dan sesudahnya dalam segmen waktu yang sama. Dengan demikian, gap yang berada di awal atau akhir segmen tidak dimasukkan.
- Mengambil informasi gap yang memenuhi kriteria dan menambahkan nama variabel target.
- Menggabungkan hasil identifikasi seluruh variabel target menggunakan `pd.concat()` ke dalam `actual_gap_df`.

Batasan:

- Identifikasi hanya mencakup tujuh variabel target yang telah ditentukan sebelumnya.
- Gap dihitung berdasarkan urutan pengamatan per jam dalam satu stasiun. Jeda waktu yang tidak tepat satu jam dianggap sebagai batas segmen baru.
- Hanya gap yang memiliki nilai teramati di kedua sisi (left and right observations) dalam segmen yang sama yang dimasukkan.
- Gap di awal atau akhir segmen tidak disertakan karena tidak memiliki pasangan nilai pengamatan di salah satu sisinya.
- Panjang gap dihitung berdasarkan jumlah baris pengamatan yang hilang secara berurutan, bukan berdasarkan selisih waktu antara nilai awal dan akhir.
- Identifikasi ini hanya memetakan pola kehilangan data aktual dan belum melakukan pengisian nilai maupun evaluasi metode imputasi.
- Gap yang teridentifikasi belum tentu seluruhnya dapat digunakan dalam simulasi karena penerapannya masih bergantung pada ketersediaan data ground truth yang lengkap.

In [5]:
def identify_actual_gaps(data, target):

    d = (
        data[["station", "datetime", target]]
        .sort_values(["station", "datetime"])
        .reset_index(drop=True)
    )

    missing = d[target].isna()

    new_seg = (
        (d["station"] != d["station"].shift())
        | (d["datetime"].diff() != pd.Timedelta(hours=1))
    )
    seg_id = new_seg.cumsum()

    run_id = (new_seg | (missing != missing.shift())).cumsum()

    runs = (
        d.assign(missing=missing, seg=seg_id)
        .groupby(run_id)
        .agg(
            station=("station", "first"),
            start_time=("datetime", "first"),
            end_time=("datetime", "last"),
            gap_length=("datetime", "size"),
            missing=("missing", "first"),
            seg=("seg", "first"),
        )
    )

    inside = (
        (runs["seg"] == runs["seg"].shift())
        & (runs["seg"] == runs["seg"].shift(-1))
    )

    gaps = runs.loc[
        runs["missing"] & inside,
        ["station", "start_time", "end_time", "gap_length"]
    ].reset_index(drop=True)

    gaps.insert(0, "target", target)

    return gaps


actual_gap_df = pd.concat(
    [identify_actual_gaps(ground_truth_data, t) for t in target_columns],
    ignore_index=True
)

## Kategorisasi dan Distribusi Gap Aktual

Tujuan:

Mengelompokkan gap aktual berdasarkan panjang durasi kehilangan data agar pola missing values dapat dianalisis secara lebih terstruktur. Hasil pengelompokan digunakan untuk mengetahui jumlah dan karakteristik gap pada setiap variabel target serta menjadi acuan dalam menentukan skenario simulasi imputasi.

Yang dilakukan:

- Membuat fungsi `categorize_gap()` untuk mengelompokkan panjang gap berdasarkan kategori yang telah ditentukan pada konfigurasi eksperimen.
- Menggunakan batas kategori 1–3 jam, 4–6 jam, 7–12 jam, 13–24 jam, 25–72 jam, dan lebih dari 72 jam.
- Menerapkan fungsi `categorize_gap()` pada kolom `gap_length` untuk menghasilkan kolom baru `gap` yang berisi kategori masing-masing gap.
- Mengelompokkan data berdasarkan kombinasi variabel target (`target`) dan kategori gap (`gap`).
- Menghitung beberapa statistik untuk setiap kelompok, yaitu:
  - `total_gaps`: jumlah blok gap yang ditemukan.
  - `total_missing_hours`: total jam pengamatan yang hilang.
  - `mean_gap_length`: rata-rata panjang gap.
  - `max_gap_length`: panjang gap maksimum.
- Menyimpan hasil agregasi ke dalam `actual_gap_summary` dan menampilkan tabel distribusi dengan pembulatan dua angka desimal.

Batasan:

- Kategorisasi hanya mencakup rentang panjang gap yang telah ditentukan dalam `gap_scenarios`. Gap di luar rentang tersebut akan diberi label `Outside range`.
- Statistik dihitung berdasarkan gap yang telah lolos kriteria identifikasi sebelumnya, yaitu memiliki nilai pengamatan di kedua sisi dalam segmen yang sama.
- Jumlah gap menunjukkan banyaknya blok kehilangan data, bukan jumlah baris yang hilang.
- Total missing hours menunjukkan akumulasi durasi kehilangan data, bukan durasi waktu kalender secara keseluruhan.
- Rata-rata dan panjang gap maksimum hanya menggambarkan distribusi gap aktual, bukan keberhasilan metode imputasi.
- Distribusi gap aktual belum menjamin seluruh kategori memiliki jumlah sampel yang cukup untuk simulasi dan evaluasi metode imputasi.

In [6]:
def categorize_gap(length):
    for category, (low, high) in gap_scenarios.items():
        if low <= length <= high:
            return category
    return "Outside range"


actual_gap_df["gap"] = actual_gap_df["gap_length"].apply(categorize_gap)

actual_gap_summary = (
    actual_gap_df
    .groupby(["target", "gap"])
    .agg(
        total_gaps=("gap_length", "count"),
        total_missing_hours=("gap_length", "sum"),
        mean_gap_length=("gap_length", "mean"),
        max_gap_length=("gap_length", "max"),
    )
    .reset_index()
)

print("\nDistribusi gap aktual:")
display(actual_gap_summary.round(2))


Distribusi gap aktual:


,target,gap,total_gaps,total_missing_hours,mean_gap_length,max_gap_length
0,DEWP,1-3 jam,210,242,1.15,3
1,DEWP,4-6 jam,26,154,5.92,6
2,DEWP,7-12 jam,1,7,7.00,7
3,PM10,1-3 jam,2045,2687,1.31,3
4,PM10,13-24 jam,62,1068,17.23,24
5,PM10,25-72 jam,21,841,40.05,64
6,PM10,4-6 jam,141,658,4.67,6
7,PM10,7-12 jam,48,407,8.48,12
8,PM10,>72 jam,4,788,197.00,343
9,PM2.5,1-3 jam,2474,3454,1.40,3


Distribusi gap aktual menunjukkan bahwa sebagian besar missing value pada variabel meteorologi (`DEWP`, `PRES`, `RAIN`, `TEMP`, dan `WSPM`) memiliki durasi pendek, yaitu 1–6 jam. Sementara itu,   `PM2.5` dan `PM10` memiliki pola gap yang lebih bervariasi, mulai dari 1–3 jam hingga lebih dari 72 jam. Gap terpanjang mencapai 343 jam pada kedua variabel. Hal ini menunjukkan bahwa penanganan missing value perlu mempertimbangkan panjang gap karena karakteristik kehilangan data setiap variabel berbeda.

## Identifikasi Pola Co-missing Aktual Antarstasiun

Tujuan:

Mengidentifikasi pola kehilangan data yang terjadi secara bersamaan (co-missing) antarstasiun pada waktu pengamatan yang sama. Informasi ini digunakan untuk membentuk pola simulasi yang lebih mendekati kondisi kehilangan data aktual, khususnya ketika suatu gap pada satu stasiun disertai kehilangan data pada stasiun lainnya.

Yang dilakukan:

- Membuat dictionary `miss_mats` untuk menyimpan matriks pola kehilangan data setiap variabel target.
- Melakukan iterasi pada seluruh variabel target yang telah ditentukan.
- Mengubah struktur data menggunakan `set_index()` dan `unstack()` sehingga waktu pengamatan menjadi baris dan masing-masing stasiun menjadi kolom.
- Mengidentifikasi nilai yang hilang menggunakan `isna()`, sehingga menghasilkan matriks Boolean (`True` untuk nilai hilang dan `False` untuk nilai tersedia).
- Menggunakan `fill_value=True` pada `unstack()` untuk menandai kombinasi waktu dan stasiun yang tidak memiliki baris pengamatan sebagai kondisi hilang.
- Mengurutkan matriks berdasarkan waktu menggunakan `sort_index()` agar urutan pengamatan tetap kronologis.
- Menyimpan indeks waktu, daftar stasiun, dan matriks Boolean ke dalam `miss_mats` untuk setiap variabel target.
- Membuat fungsi `get_template()` untuk mengambil pola kehilangan data antarstasiun berdasarkan waktu awal (`start_time`) dan waktu akhir (`end_time`) suatu gap aktual.
- Mengambil posisi waktu awal dan akhir menggunakan `get_loc()`, kemudian mengekstrak bagian matriks yang sesuai dengan rentang waktu tersebut.
- Menghasilkan template berupa daftar stasiun dan array Boolean berdimensi panjang gap × jumlah stasiun.

Batasan:

- Identifikasi pola co-missing hanya dilakukan pada tujuh variabel target yang telah ditentukan.
- Pola yang dihasilkan berdasarkan kondisi kehilangan data pada dataset asli, bukan pola kehilangan data yang dibuat secara artifisial.
- Template hanya mencakup rentang waktu gap aktual yang diberikan, sehingga pola kehilangan data di luar rentang tersebut tidak ikut dianalisis.
- Penggunaan `fill_value=True` mengasumsikan bahwa kombinasi waktu dan stasiun yang tidak memiliki baris pengamatan merupakan kondisi kehilangan data. Hal ini dapat mencampurkan nilai yang benar-benar hilang dengan waktu ketika stasiun tidak memiliki catatan pengamatan.
- Template menggambarkan kesamaan waktu kehilangan data antarstasiun, tetapi tidak menjelaskan penyebab terjadinya kehilangan data tersebut.
- Pola co-missing yang diperoleh belum tentu dapat diterapkan pada seluruh simulasi karena bergantung pada ketersediaan stasiun dan kelengkapan data pada rentang waktu yang bersangkutan.

In [7]:
miss_mats = {}

for t in target_columns:
    m = (
        ground_truth_data
        .set_index(["datetime", "station"])[t]
        .isna()
        .unstack("station", fill_value=True)
        .sort_index()
    )
    miss_mats[t] = (m.index, list(m.columns), m.to_numpy(dtype=bool))


def get_template(target, start_time, end_time):
    idx, cols, arr = miss_mats[target]
    i0 = idx.get_loc(start_time)
    i1 = idx.get_loc(end_time)
    return cols, arr[i0:i1 + 1, :]

## Generator Artificial Missingness

Tujuan:

Membuat simulasi kehilangan data secara artifisial (artificial missingness) berdasarkan karakteristik gap aktual yang ditemukan pada dataset asli. Simulasi ini digunakan untuk menguji dan membandingkan kinerja berbagai metode imputasi pada kondisi data yang nilai aslinya masih diketahui (ground truth).

Yang dilakukan:

- Membuat fungsi `build_sequence_cache()` untuk mengidentifikasi rangkaian waktu pengamatan yang berurutan selama satu jam pada setiap stasiun.
- Mengelompokkan data berdasarkan stasiun menggunakan `groupby()`, kemudian mengambil posisi baris dan informasi waktunya.
- Mengidentifikasi jeda waktu yang tidak berurutan satu jam menggunakan `np.diff()` dan `np.where()`.
- Memisahkan rangkaian pengamatan berdasarkan jeda waktu menggunakan `np.split()`. Hasilnya disimpan dalam `cache` agar pencarian lokasi simulasi gap lebih efisien.
- Membuat fungsi `create_actual_gap_mask()` untuk menghasilkan mask Boolean yang menentukan posisi nilai yang akan dibuat hilang dalam simulasi.
- Mengambil kumpulan gap aktual berdasarkan variabel target dan kategori panjang gap yang sedang diuji.
- Memilih sampel gap aktual secara acak menggunakan `rng.choice()`. Jika jumlah gap aktual lebih sedikit daripada jumlah blok yang diminta, pemilihan dilakukan dengan pengembalian (sampling with replacement).
- Mempertahankan identitas stasiun dan panjang gap dari sampel aktual, tetapi menempatkan blok kehilangan data pada posisi waktu yang dipilih secara acak dalam rangkaian pengamatan kontinu stasiun yang sama.
- Memastikan rangkaian memiliki minimal satu pengamatan sebelum dan sesudah blok gap agar simulasi memiliki nilai tetangga yang tersedia.
- Memeriksa kemungkinan tumpang tindih atau blok yang saling menempel dengan menggunakan area pemeriksaan yang mencakup satu baris sebelum dan sesudah blok.
- Membatasi percobaan penempatan maksimal 50 kali untuk setiap blok agar proses tidak terus mencari lokasi yang tidak tersedia.
- Mengubah posisi blok terpilih menjadi `True` pada `mask` dan menyimpan panjang blok ke `placed_lengths`.
- Menyimpan informasi penempatan berupa posisi blok, stasiun, serta waktu awal dan akhir gap asal ke dalam `placements`. Informasi ini nantinya digunakan untuk membentuk simulasi co-missing.
- Mengembalikan tiga keluaran, yaitu `mask`, `placed_lengths`, dan `placements`.

Batasan:

- Simulasi hanya menggunakan karakteristik gap aktual yang telah berhasil diidentifikasi dan dikategorikan sebelumnya.
- Panjang gap dan identitas stasiun dipertahankan berdasarkan sampel aktual, tetapi posisi penempatannya dipilih secara acak sehingga tidak mempertahankan waktu kejadian aslinya.
- Penempatan hanya dilakukan pada rangkaian pengamatan yang kontinu dan memiliki nilai tetangga di kedua sisi.
- Blok gap tidak boleh tumpang tindih atau saling menempel dengan blok yang sudah ditempatkan dalam satu simulasi.
- Jumlah blok yang berhasil ditempatkan dapat lebih sedikit daripada jumlah yang diminta karena keterbatasan lokasi yang memenuhi kriteria.
- Jika jumlah gap aktual lebih sedikit daripada kebutuhan simulasi, pengambilan sampel dapat menggunakan gap yang sama lebih dari satu kali.
- Percobaan penempatan dibatasi maksimal 50 kali per blok. Dengan demikian, kegagalan penempatan tidak selalu berarti tidak ada lokasi yang valid, tetapi bisa terjadi karena batas percobaan.
- Penggunaan `mask` yang sama untuk kedua skenario memungkinkan perbandingan yang lebih adil karena posisi gap utama tidak berubah berdasarkan skenario.
- Simulasi ini merepresentasikan kehilangan data berdasarkan panjang gap aktual, tetapi belum sepenuhnya merepresentasikan mekanisme penyebab kehilangan data yang sebenarnya.

In [8]:
def build_sequence_cache(data):
    cache = {}
    for station, group in data.groupby("station"):
        pos = group.index.to_numpy()
        ts = group["datetime"].to_numpy()
        breaks = np.where(np.diff(ts) != np.timedelta64(1, "h"))[0] + 1
        cache[station] = np.split(pos, breaks)
    return cache


def create_actual_gap_mask(data, target, actual_gap_df, gap_name,
                           rng, n_blocks, seq_cache):

    mask = np.zeros(len(data), dtype=bool)
    placed_lengths = []
    placements = []

    pool = actual_gap_df[
        (actual_gap_df["target"] == target)
        & (actual_gap_df["gap"] == gap_name)
    ]

    if pool.empty:
        return mask, placed_lengths, placements

    idx = rng.choice(
        len(pool),
        size=n_blocks,
        replace=len(pool) < n_blocks
    )
    sampled = pool.iloc[idx]

    for station, gap_length, st, et in zip(
        sampled["station"], sampled["gap_length"],
        sampled["start_time"], sampled["end_time"]
    ):

        gap_length = int(gap_length)

        valid = [
            s for s in seq_cache.get(station, [])
            if len(s) >= gap_length + 2
        ]

        if not valid:
            continue

        for _ in range(50):

            seq = valid[rng.integers(len(valid))]
            start = int(rng.integers(1, len(seq) - gap_length))

            surrounding = seq[start - 1:start + gap_length + 1]

            if mask[surrounding].any():
                continue

            block = seq[start:start + gap_length]
            mask[block] = True
            placed_lengths.append(gap_length)
            placements.append((block.copy(), station, st, et))
            break

    return mask, placed_lengths, placements

## Pembentukan Context Mask untuk Skenario Co-missing

Tujuan:

Membentuk kehilangan data tambahan pada stasiun lain berdasarkan pola co-missing aktual yang telah diidentifikasi sebelumnya. Mask tambahan ini digunakan untuk mensimulasikan kondisi ketika beberapa stasiun mengalami kehilangan data secara bersamaan, sehingga dapat menguji kemampuan metode imputasi dalam memanfaatkan informasi dari stasiun lain. Nilai yang disembunyikan sebagai konteks tidak ikut dievaluasi.

Yang dilakukan:

- Membuat fungsi `build_context_mask()` untuk menghasilkan context mask, yaitu mask Boolean yang menandai nilai tambahan yang akan disembunyikan pada stasiun lain.
- Menginisialisasi `ctx` sebagai array Boolean bernilai `False` dengan panjang yang sama dengan data.
- Mengambil informasi waktu dari `base` untuk mencocokkan posisi pengamatan berdasarkan stasiun dan waktu.
- Melakukan iterasi terhadap `placements` yang berisi informasi blok gap hasil simulasi, meliputi posisi blok, stasiun, waktu awal, dan waktu akhir gap aktual asal.
- Mengambil template pola co-missing menggunakan `get_template()` berdasarkan variabel target dan rentang waktu gap aktual.
- Memeriksa kesesuaian panjang template dengan panjang blok simulasi. Jika tidak sesuai, penempatan tersebut dilewati.
- Menggunakan waktu aktual pada posisi blok simulasi sebagai acuan untuk menerapkan pola kehilangan data antarstasiun.
- Mengidentifikasi stasiun lain yang memiliki nilai hilang pada setiap posisi waktu berdasarkan template.
- Mengecualikan stasiun tempat blok evaluasi berada agar kehilangan data utama tidak dihitung kembali sebagai kehilangan data konteks.
- Mengumpulkan pasangan identitas stasiun dan waktu yang akan disembunyikan sebagai konteks.
- Membentuk `MultiIndex` dari pasangan stasiun dan waktu, kemudian mencocokkannya dengan indeks data menggunakan `get_indexer()`.
- Menandai posisi yang cocok sebagai `True` pada `ctx`, sehingga nilai tersebut menjadi kehilangan data tambahan.
- Membentuk daftar posisi yang tidak boleh disembunyikan berdasarkan stasiun dan waktu pada blok evaluasi, termasuk satu jam sebelum dan sesudahnya.
- Menghapus tanda kehilangan data konteks pada posisi yang termasuk dalam daftar terlarang menggunakan `ctx &= ~key_index.isin(forbidden)`.
- Mengembalikan `ctx` sebagai mask tambahan yang terpisah dari mask evaluasi utama.

Batasan:

- Pembentukan context mask hanya mengikuti pola co-missing aktual yang tersedia pada dataset asli.
- Kehilangan data tambahan hanya diterapkan pada stasiun lain, bukan pada stasiun tempat blok evaluasi berada.
- Pola kehilangan data mengikuti posisi temporal dalam blok, bukan sekadar jumlah stasiun yang mengalami kehilangan data.
- Penempatan konteks bergantung pada kecocokan pasangan stasiun dan waktu. Jika pasangan tersebut tidak ditemukan dalam indeks data, posisi tersebut tidak akan ditandai sebagai hilang.
- Template yang panjangnya tidak sesuai dengan blok simulasi akan dilewati sehingga tidak semua penempatan menghasilkan konteks tambahan.
- Sel pada stasiun dan waktu yang sama dengan blok evaluasi, termasuk satu jam sebelum dan sesudahnya, tidak disembunyikan untuk menjaga ketersediaan nilai tetangga.
- Nilai yang disembunyikan melalui `ctx` tidak termasuk dalam evaluasi RMSE maupun MAE. Evaluasi hanya dilakukan terhadap nilai yang ditandai oleh mask utama.
- Mask ini hanya mensimulasikan kondisi kehilangan data tambahan dan tidak membuktikan bahwa pola kehilangan data antarstasiun memiliki penyebab yang sama.

Catatan: `ctx` dan `mask` memiliki fungsi berbeda. `mask` menentukan nilai yang sengaja dihilangkan dan dievaluasi, sedangkan `ctx` menentukan nilai tambahan yang dihilangkan hanya untuk membentuk kondisi input skenario co-missing.

In [9]:
def build_context_mask(base, key_index, mask, placements, target):

    ctx = np.zeros(len(base), dtype=bool)
    base_times = base["datetime"].to_numpy()

    key_stations, key_times = [], []

    for block_pos, station, st, et in placements:

        cols, tmpl = get_template(target, st, et)

        if tmpl.shape[0] != len(block_pos):
            continue

        times = base_times[block_pos]

        for j, col in enumerate(cols):
            if col == station:
                continue
            hit = tmpl[:, j]
            if not hit.any():
                continue
            key_stations.append(np.full(int(hit.sum()), col, dtype=object))
            key_times.append(times[hit])

    if key_stations:
        keys = pd.MultiIndex.from_arrays([
            np.concatenate(key_stations),
            np.concatenate(key_times),
        ])
        pos = key_index.get_indexer(keys)
        ctx[pos[pos >= 0]] = True

    eval_rows = base.loc[mask, ["station", "datetime"]]
    eval_stations = eval_rows["station"].to_numpy()
    eval_times = eval_rows["datetime"].to_numpy()

    forbidden = pd.MultiIndex.from_arrays([
        np.concatenate([eval_stations] * 3),
        np.concatenate([eval_times + np.timedelta64(h, "h") for h in (-1, 0, 1)]),
    ])
    ctx &= ~key_index.isin(forbidden)

    return ctx

## Implementasi Metode Dasar Imputasi

Tujuan:

Mengimplementasikan beberapa metode imputasi sebagai kandidat pembanding untuk menangani missing values pada setiap variabel target. Metode yang digunakan memiliki pendekatan berbeda, yaitu interpolasi berdasarkan urutan waktu, pengisian berdasarkan median kondisional, dan imputasi berdasarkan kemiripan karakteristik data menggunakan K-Nearest Neighbors (KNN).

Yang dilakukan:

- Membuat fungsi `interpolate_method()` untuk melakukan interpolasi linear berbasis waktu pada setiap stasiun.
    - Memisahkan data berdasarkan stasiun menggunakan `groupby()`.
    - Mengurutkan pengamatan berdasarkan `datetime`.
    - Menggunakan `set_index()` untuk menjadikan waktu sebagai indeks.
    - Melakukan interpolasi dengan `method="time"` agar perhitungan mempertimbangkan jarak waktu antarobservasi.
    - Menggunakan `limit_area="inside"` agar interpolasi hanya dilakukan pada gap yang memiliki nilai pengamatan di kedua sisi.
    - Menggabungkan hasil interpolasi ke dalam kolom target tanpa mengubah data asli.

- Membuat fungsi `median_method()` untuk mengisi nilai hilang menggunakan median berdasarkan kombinasi bulan dan jam pengamatan.
    - Mengambil data yang tidak disembunyikan menggunakan `~hidden` sebagai data pelatihan.
    - Menghitung median target berdasarkan kelompok `month` dan `hour`.
    - Menghitung median global sebagai nilai pengganti apabila median berdasarkan kelompok tidak tersedia.
    - Mengambil kombinasi bulan dan jam pada posisi yang akan dievaluasi.
    - Mencocokkan nilai median berdasarkan kombinasi tersebut dan menggunakan median global apabila hasilnya kosong.
    - Mengisi posisi yang ditandai oleh `mask` dengan nilai median yang diperoleh.

- Membuat fungsi `knn_method()` untuk melakukan imputasi menggunakan KNN dengan fitur meteorologi sebagai variabel pendukung.
    - Memisahkan data menjadi data pelatihan (`train_data`) dan data pengujian (`test_data`) berdasarkan `hidden` dan `mask`.
    - Mengambil sampel data pelatihan maksimal sebanyak 3.000 baris menggunakan `sample()` agar proses komputasi lebih efisien.
    - Melakukan standardisasi fitur meteorologi menggunakan `StandardScaler`. Parameter standardisasi dipelajari dari data pelatihan dan diterapkan pada data pengujian.
    - Menggabungkan nilai target dan fitur meteorologi yang telah distandardisasi ke dalam matriks pelatihan.
    - Membuat matriks pengujian dengan nilai target berupa `NaN`, sedangkan fitur meteorologi tetap tersedia.
    - Menginisialisasi `KNNImputer` dengan lima tetangga terdekat (`n_neighbors=5`) dan pembobotan berdasarkan jarak (`weights="distance"`).
    - Melakukan proses imputasi pada data pengujian secara bertahap menggunakan ukuran batch maksimal 1.000 baris.
    - Mengambil hasil imputasi dari kolom target dan memasukkannya ke posisi yang ditandai oleh `mask`.

Batasan:

- Interpolasi linear:
    - Hanya bekerja pada gap yang memiliki nilai pengamatan di kedua sisi.
    - Mengasumsikan perubahan nilai antarwaktu dapat didekati dengan hubungan linear.
    - Dilakukan secara terpisah pada setiap stasiun sehingga tidak memanfaatkan informasi dari stasiun lain.
    - Tidak dapat mengisi gap di awal atau akhir rangkaian pengamatan.

- Median kondisional:
    - Hanya mempertimbangkan kombinasi bulan dan jam, tanpa memperhitungkan urutan waktu maupun karakteristik stasiun.
    - Tidak menangkap perubahan nilai yang terjadi secara spesifik pada setiap stasiun.
    - Penggunaan median global sebagai fallback dapat mengurangi representasi karakteristik kelompok yang sebenarnya.

- KNN Imputer:
    - Menggunakan lima tetangga terdekat dengan fitur meteorologi sebagai dasar pengukuran kemiripan.
    - Tidak menggunakan identitas stasiun maupun informasi bulan dan jam sebagai fitur.
    - Sampel pelatihan dibatasi maksimal 3.000 baris sehingga tidak seluruh data pelatihan digunakan.
    - Hasil imputasi bergantung pada ketersediaan dan kualitas fitur meteorologi.
    - Standardisasi membantu menyamakan skala fitur, tetapi tidak menjamin seluruh fitur memiliki kontribusi yang sama terhadap kemiripan.
    - Proses imputasi dilakukan per batch untuk mengurangi beban komputasi.

Catatan: Ketiga metode memiliki pendekatan dan asumsi yang berbeda. Oleh karena itu, hasilnya perlu dibandingkan menggunakan data simulasi yang sama agar metode terbaik dapat dipilih berdasarkan kinerja evaluasi, bukan hanya berdasarkan asumsi atau kompleksitas metode.

In [10]:
def interpolate_method(data, target):
    result = data[target].copy()

    for station, group in data.groupby("station"):

        station_data = group.sort_values("datetime")

        interpolated = (
            station_data
            .set_index("datetime")[target]
            .interpolate(method="time", limit_area="inside")
        )

        result.loc[station_data.index] = interpolated.to_numpy()

    return result


def median_method(data, target, mask, hidden):
    train_data = data.loc[~hidden]

    conditional_median = (
        train_data.groupby(["month", "hour"])[target].median()
    )
    global_median = train_data[target].median()

    keys = pd.MultiIndex.from_frame(data.loc[mask, ["month", "hour"]])

    median_values = conditional_median.reindex(keys).to_numpy()
    median_values = np.where(np.isnan(median_values), global_median, median_values)

    result = data[target].copy()
    result.loc[mask] = median_values

    return result


def knn_method(data, target, mask, hidden, features):
    train_data = data.loc[~hidden]
    test_data = data.loc[mask]

    train_sample = train_data.sample(
        n=min(knn_train_size, len(train_data)),
        random_state=42
    )

    scaler = StandardScaler()

    X_train_features = scaler.fit_transform(train_sample[features])
    X_test_features = scaler.transform(test_data[features])

    X_train = np.column_stack([train_sample[target].to_numpy(), X_train_features])
    X_test = np.column_stack([np.full(len(test_data), np.nan), X_test_features])

    knn = KNNImputer(n_neighbors=5, weights="distance")
    knn.fit(X_train)

    predictions = []

    for start in range(0, len(X_test), knn_batch_size):
        batch = X_test[start:start + knn_batch_size]
        predictions.extend(knn.transform(batch)[:, 0])

    result = data[target].copy()
    result.loc[mask] = predictions

    return result

## Implementasi Metode Imputasi Berbasis Stasiun Lain (Neighbor-Based Methods)

Tujuan:

Mengembangkan metode imputasi yang memanfaatkan informasi kualitas udara dari stasiun pengamatan lain untuk memperkirakan nilai yang hilang. Metode ini digunakan untuk menguji apakah informasi antarstasiun dapat meningkatkan hasil imputasi dibandingkan metode yang hanya mengandalkan data dari stasiun itu sendiri. Terdapat tiga pendekatan yang dibandingkan, yaitu Neighbor Station, Interpolation + Neighbor Correction (Level), dan Interpolation + Neighbor Correction (Anomaly).

Yang dilakukan:

- Membuat fungsi `neighbor_methods()` yang menghasilkan tiga prediksi imputasi berbasis informasi antarstasiun.
- Mengubah struktur data menjadi matriks dengan waktu sebagai baris dan stasiun sebagai kolom menggunakan `set_index()` dan `unstack()`.
- Mengidentifikasi nilai yang tersedia menggunakan `np.isfinite()` untuk memastikan nilai yang disembunyikan tidak digunakan dalam perhitungan.
- Menghitung jumlah dan total nilai pengamatan yang tersedia pada setiap waktu menggunakan `count` dan `total_l`.
- Menghitung rata-rata nilai setiap stasiun (`mu`) berdasarkan data yang tidak disembunyikan, kemudian menghitung nilai anomali (`vals_a`) sebagai selisih antara nilai pengamatan dan rata-rata stasiunnya.
- Menghitung total nilai anomali yang tersedia pada setiap waktu untuk digunakan dalam pendekatan berbasis anomali.
- Mencocokkan indeks waktu dan stasiun pada matriks dengan data awal menggunakan `get_indexer()`.
- Menginisialisasi hasil prediksi untuk ketiga metode dan membuat penanda `used` untuk mencatat sel evaluasi yang berhasil menggunakan informasi dari stasiun lain.

- Membuat fungsi internal `fit_and_correct()` untuk menghitung hubungan linear antara informasi stasiun lain dan nilai target pada stasiun yang sedang diuji.
    - Memilih pasangan nilai yang tersedia untuk pelatihan.
    - Menggunakan `np.polyfit()` untuk memperoleh kemiringan (slope) dan intersep regresi linear.
    - Membentuk interpolasi nilai stasiun sendiri berdasarkan titik pengamatan yang tersedia.
    - Menghitung koreksi berdasarkan selisih antara informasi stasiun lain dan hasil interpolasi stasiun sendiri.
    - Menghasilkan koreksi bernilai nol apabila informasi pendukung tidak tersedia.

- Mengimplementasikan tiga metode imputasi:
    1. Neighbor Station: menggunakan regresi linear antara nilai target pada suatu stasiun dan rata-rata nilai stasiun lain pada waktu yang sama. Jika informasi stasiun lain tidak tersedia atau data pelatihan tidak mencukupi, digunakan prediksi median sebagai cadangan.
    2. Interp + NC (level): menggunakan hasil interpolasi linear stasiun sendiri sebagai nilai dasar, kemudian menambahkan koreksi berdasarkan perbedaan level rata-rata stasiun lain terhadap interpolasi tersebut.
    3. Interp + NC (anomaly): menggunakan interpolasi linear sebagai nilai dasar, kemudian stasiun lain dinyatakan sebagai anomali terhadap rata-rata stasiunnya masing-masing (dihitung hanya dari nilai yang tidak disembunyikan), sehingga perubahan komposisi stasiun yang teramati tidak mengubah level rata-rata. Pendekatan ini bertujuan mengurangi pengaruh perbedaan level dasar antarstasiun.

- Membatasi hasil prediksi agar tidak negatif untuk variabel yang telah ditentukan dalam `nonneg_targets` menggunakan `np.clip()`.
- Menggunakan median sebagai metode cadangan untuk Neighbor Station apabila informasi stasiun lain tidak tersedia atau jumlah data pelatihan kurang dari 10 pasangan observasi.
- Menggunakan interpolasi linear sebagai metode cadangan untuk dua metode Interpolation + Neighbor Correction apabila hasil prediksinya tidak tersedia.
- Menghitung statistik ketersediaan informasi antarstasiun pada posisi yang dievaluasi, meliputi:
    - `fallback_rate`: proporsi sel evaluasi yang menggunakan prediksi cadangan pada metode Neighbor Station.
    - `mean_share_others_hidden`: rata-rata proporsi stasiun lain yang nilainya tidak tersedia.
    - `share_all_others_hidden`: proporsi sel evaluasi ketika seluruh stasiun lain tidak memiliki nilai pengamatan.
- Mengembalikan tiga hasil prediksi dalam bentuk `Series` beserta statistik ketersediaan informasi antarstasiun.

Batasan:

- Metode hanya memanfaatkan informasi dari stasiun lain pada waktu pengamatan yang sama.
- Nilai yang disembunyikan tidak digunakan sebagai informasi pelatihan karena perhitungan hanya menggunakan nilai yang tersedia.
- Hubungan antara nilai suatu stasiun dan stasiun lain diasumsikan dapat didekati menggunakan regresi linear.
- Regresi linear hanya dilakukan apabila tersedia minimal 10 pasangan observasi yang lengkap. Jika kurang, metode Neighbor Station menggunakan median sebagai cadangan.
- Metode Neighbor Station tidak secara langsung mempertimbangkan urutan waktu atau pola perubahan temporal pada stasiun target.
- Metode Interpolation + NC tetap bergantung pada interpolasi linear sehingga memiliki keterbatasan ketika menghadapi gap panjang atau perubahan nilai yang tidak linear.
- Pendekatan level dapat dipengaruhi oleh perbedaan karakteristik dasar antarstasiun dan perubahan jumlah stasiun yang tersedia.
- Pendekatan anomaly menggunakan rata-rata setiap stasiun yang dihitung dari nilai yang tidak disembunyikan, sehingga hasilnya bergantung pada ketersediaan data masing-masing stasiun.
- Jika informasi stasiun lain tidak tersedia, koreksi pada metode Interpolation + NC menjadi nol sehingga hasilnya kembali ke interpolasi biasa.
- Pembatasan nilai negatif hanya diterapkan pada variabel yang tercantum dalam `nonneg_targets`.
- Ketiga metode menghasilkan prediksi berdasarkan pendekatan yang berbeda, sehingga keunggulannya tetap perlu dibuktikan melalui evaluasi menggunakan RMSE dan MAE.

Catatan evaluasi kode: Ada satu hal yang perlu diperhatikan. Pada `neighbor_stats`, perhitungan `avail = count[time_idx[mask]]` menghitung jumlah seluruh stasiun yang memiliki nilai pada waktu tersebut, bukan secara eksplisit jumlah stasiun lain yang tersedia. Karena sel target yang dievaluasi memang hilang, perhitungan ini setara dengan jumlah stasiun lain yang tersedia hanya jika nilai target tersebut benar-benar `NaN` pada matriks `wide`. Selain itu, statistik tersebut dihitung untuk seluruh sel evaluasi, bukan hanya sel yang berhasil diprediksi menggunakan tetangga.

In [11]:
def neighbor_methods(data, target, mask, median_pred, interp_pred):
    wide = data.set_index(["datetime", "station"])[target].unstack("station")
    vals = wide.to_numpy(dtype=float)
    finite = np.isfinite(vals)
    n_st = vals.shape[1]

    count = finite.sum(axis=1)
    total_l = np.where(finite, vals, 0.0).sum(axis=1)

    col_cnt = finite.sum(axis=0)
    mu = np.where(
        col_cnt > 0,
        np.where(finite, vals, 0.0).sum(axis=0) / np.maximum(col_cnt, 1),
        0.0
    )
    vals_a = vals - mu[None, :]
    total_a = np.where(finite, vals_a, 0.0).sum(axis=1)

    time_idx = wide.index.get_indexer(data["datetime"])
    station_arr = data["station"].to_numpy()
    datetimes = data["datetime"].to_numpy()
    y_all = data[target].to_numpy(dtype=float)
    interp_all = interp_pred.to_numpy(dtype=float)

    res_nb = y_all.copy()
    res_cl = y_all.copy()
    res_ca = y_all.copy()
    used = np.zeros(len(data), dtype=bool)

    def fit_and_correct(x, y, dt_index):
        train = np.isfinite(x) & np.isfinite(y)
        if train.sum() < 10:
            return None
        slope, intercept = np.polyfit(x[train], y[train], 1)

        x_edges = np.where(np.isfinite(y), x, np.nan)
        x_interp = (
            pd.Series(x_edges, index=dt_index)
            .interpolate(method="time", limit_area="inside")
            .to_numpy()
        )
        corr = np.where(
            np.isfinite(x) & np.isfinite(x_interp),
            slope * (x - x_interp),
            0.0
        )
        return slope, intercept, corr

    for j, station in enumerate(wide.columns):

        sel = np.where(station_arr == station)[0]
        if len(sel) == 0:
            continue

        others_cnt = count - finite[:, j]
        den = np.maximum(others_cnt, 1)

        x_level = np.where(
            others_cnt > 0,
            (total_l - np.where(finite[:, j], vals[:, j], 0.0)) / den,
            np.nan
        )
        x_anom = np.where(
            others_cnt > 0,
            (total_a - np.where(finite[:, j], vals_a[:, j], 0.0)) / den,
            np.nan
        )

        ti = time_idx[sel]
        y = y_all[sel]
        m = mask[sel]
        dt_index = pd.DatetimeIndex(datetimes[sel])

        fit_l = fit_and_correct(x_level[ti], y, dt_index)
        if fit_l is not None:
            slope, intercept, corr = fit_l
            xl = x_level[ti]

            ok = m & np.isfinite(xl)
            pred = intercept + slope * xl[ok]
            if target in nonneg_targets:
                pred = np.clip(pred, 0, None)
            res_nb[sel[ok]] = pred
            used[sel[ok]] = True

            comb = interp_all[sel] + corr
            if target in nonneg_targets:
                comb = np.clip(comb, 0, None)
            res_cl[sel[m]] = comb[m]

        fit_a = fit_and_correct(x_anom[ti], y, dt_index)
        if fit_a is not None:
            _, _, corr_a = fit_a
            comb_a = interp_all[sel] + corr_a
            if target in nonneg_targets:
                comb_a = np.clip(comb_a, 0, None)
            res_ca[sel[m]] = comb_a[m]

    need_fallback = mask & ~used
    res_nb[need_fallback] = median_pred.to_numpy(dtype=float)[need_fallback]

    for res in (res_cl, res_ca):
        unfilled = mask & ~np.isfinite(res)
        res[unfilled] = interp_all[unfilled]

    n_eval = max(int(mask.sum()), 1)
    avail = count[time_idx[mask]]
    others_total = max(n_st - 1, 1)
    neighbor_stats = {
        "fallback_rate": need_fallback.sum() / n_eval,
        "mean_share_others_hidden": float(np.mean(1 - avail / others_total)),
        "share_all_others_hidden": float(np.mean(avail == 0)),
    }

    return (
        pd.Series(res_nb, index=data.index),
        pd.Series(res_cl, index=data.index),
        pd.Series(res_ca, index=data.index),
        neighbor_stats,
    )

## Fungsi Metrik Evaluasi Regresi

Tujuan

Mengukur tingkat kesalahan dan karakteristik hasil prediksi dari setiap metode imputasi regresi dengan membandingkan nilai hasil imputasi terhadap nilai aktual (ground truth). Evaluasi ini dilakukan untuk mengetahui seberapa akurat metode dalam memperkirakan nilai yang hilang serta apakah hasil imputasinya cenderung mengalami bias atau perubahan variasi data.

Yang Dilakukan

- Menghitung Root Mean Squared Error (RMSE) untuk mengukur rata-rata kesalahan prediksi dengan memberikan penalti lebih besar pada kesalahan yang bernilai tinggi.
- Menghitung Mean Absolute Error (MAE) untuk mengukur rata-rata selisih absolut antara nilai prediksi dan nilai aktual.
- Menghitung Mean Absolute Percentage Error (MAPE) untuk mengukur rata-rata kesalahan prediksi dalam bentuk persentase. Perhitungan hanya menggunakan nilai aktual yang tidak sama dengan nol.
- Menghitung Bias untuk mengetahui kecenderungan hasil prediksi, apakah cenderung lebih tinggi atau lebih rendah dibandingkan nilai aktual.
- Menghitung Standard Deviation Ratio untuk membandingkan tingkat variasi hasil prediksi terhadap variasi nilai aktual.

Batasan

- Evaluasi hanya dilakukan pada pasangan nilai aktual dan prediksi yang tersedia, sehingga nilai yang tidak memiliki pasangan valid tidak dapat disertakan.
- MAPE tidak memperhitungkan nilai aktual nol karena dapat menyebabkan pembagian dengan nol.
- RMSE dan MAE menggunakan satuan yang sama dengan variabel yang dievaluasi, sedangkan MAPE dinyatakan dalam persentase.
- Bias mendekati nol menunjukkan tidak adanya kecenderungan kesalahan sistematis secara rata-rata, tetapi tidak menjamin setiap prediksi akurat.
- Standard Deviation Ratio mendekati 1 menunjukkan tingkat variasi hasil prediksi yang mendekati variasi nilai aktual, tetapi tidak menjamin distribusi maupun pola datanya sama.
- Metrik evaluasi ini tidak secara langsung mengukur kemampuan metode dalam mempertahankan pola temporal (time series).

Catatan: Kelima metrik digunakan secara bersamaan karena masing-masing mengukur aspek evaluasi yang berbeda. RMSE dan MAE mengukur kesalahan prediksi, MAPE mengukur kesalahan relatif, Bias mengukur kecenderungan kesalahan, sedangkan Standard Deviation Ratio mengukur kesesuaian tingkat variasi data.

In [12]:
def regression_metrics(y_true, y_pred):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)

    nz = y_true != 0
    mape = (
        np.mean(np.abs((y_true[nz] - y_pred[nz]) / y_true[nz])) * 100
        if nz.any() else np.nan
    )

    bias = float(np.mean(y_pred - y_true))

    if len(y_true) > 1 and np.std(y_true, ddof=1) > 0:
        std_ratio = float(np.std(y_pred, ddof=1) / np.std(y_true, ddof=1))
    else:
        std_ratio = np.nan

    return rmse, mae, mape, bias, std_ratio

## Simulasi dan Evaluasi Metode Imputasi

Tujuan

Melakukan simulasi artificial missing values berdasarkan karakteristik gap aktual pada dataset untuk membandingkan performa beberapa metode imputasi dalam berbagai kondisi kehilangan data. Evaluasi dilakukan pada beberapa variabel target, kategori panjang gap, skenario missingness, dan pengulangan (trials) dengan random seed berbeda agar hasil perbandingan lebih representatif.

Yang Dilakukan

- Menyiapkan data aktual (ground truth) untuk setiap variabel target beserta fitur meteorologi yang digunakan sebagai variabel pendukung imputasi.
- Membangun sequence cache untuk menyimpan informasi urutan data berdasarkan stasiun dan waktu, serta menyiapkan indeks stasiun dan waktu untuk membantu proses simulasi.
- Mengidentifikasi jumlah gap aktual yang tersedia berdasarkan variabel target dan kategori panjang gap sebagai dasar penentuan jumlah blok missing yang akan disimulasikan.
- Membuat artificial missing values menggunakan `create_actual_gap_mask()` dengan mengambil pola gap dari data aktual. Proses ini dilakukan menggunakan beberapa random seed yang dibagi menjadi fase seleksi dan konfirmasi.
- Membangun context mask untuk mengidentifikasi titik data pendukung yang ikut disembunyikan dalam skenario co-missing.
- Menjalankan simulasi dalam dua skenario:
  - Independent: hanya nilai pada blok gap yang disimulasikan sebagai missing.
  - Co-missing: nilai pada blok gap dan titik data pendukung yang berada dalam konteks simulasi disembunyikan.
- Menerapkan enam metode imputasi untuk setiap skenario, yaitu:
  1. Linear Interpolation
  2. Conditional Median
  3. KNN Imputer
  4. Neighbor Station
  5. Interp + NC (level)
  6. Interp + NC (anomaly)
- Mengambil hasil prediksi hanya pada posisi yang sengaja dibuat missing, kemudian membandingkannya dengan nilai aktual menggunakan RMSE, nRMSE, MAE, MAPE, Bias, nBias, dan Standard Deviation Ratio.
- Mencatat hasil evaluasi ke dalam `evaluation_results`, termasuk jumlah titik yang dievaluasi, jumlah blok gap yang berhasil ditempatkan, dan fallback rate untuk metode Neighbor Station.
- Mencatat pemeriksaan simulasi ke dalam `simulation_checks`, seperti jumlah gap aktual yang tersedia, jumlah blok yang diminta dan berhasil ditempatkan, jumlah titik yang disembunyikan, serta proporsi fitur pendukung lain yang ikut missing.
- Menggabungkan hasil akhir menjadi `evaluation_df` dan `simulation_check_df` untuk digunakan pada tahap analisis dan perbandingan performa metode imputasi.

Batasan

- Simulasi hanya menggunakan pola gap aktual yang tersedia pada dataset. Oleh karena itu, variasi panjang dan karakteristik gap yang dapat diuji bergantung pada ketersediaan gap aktual.
- Jumlah blok gap yang berhasil ditempatkan dapat berbeda dari jumlah yang diminta apabila jumlah gap aktual yang tersedia terbatas atau terdapat kendala penempatan.
- Skenario co-missing hanya merepresentasikan kondisi kehilangan data pendukung sesuai aturan pembentukan `context_mask`, bukan seluruh kemungkinan pola kehilangan data multivariat.
- Evaluasi hanya dilakukan pada titik yang sengaja disembunyikan dan masih memiliki nilai aktual sebagai pembanding (ground truth).
- Hasil simulasi dipengaruhi oleh pemilihan random seed, sehingga performa metode tidak dapat dinilai hanya dari satu kali percobaan.
- Pembagian fase seleksi dan konfirmasi perlu dipertahankan dalam analisis agar hasil pemilihan metode tidak tercampur dengan hasil evaluasi konfirmasi.
- Metrik evaluasi mengukur kesesuaian nilai imputasi terhadap nilai aktual, tetapi belum secara langsung menjamin bahwa pola temporal, hubungan antarvariabel, dan karakteristik distribusi data tetap terjaga.
- Hasil evaluasi berlaku pada dataset, variabel target, kategori gap, dan skenario simulasi yang diuji sehingga generalisasi ke kondisi dataset lain perlu dilakukan secara terpisah.

Catatan: `evaluation_df` digunakan sebagai tabel utama hasil perbandingan performa metode imputasi, sedangkan `simulation_check_df` digunakan untuk memeriksa apakah proses simulasi berjalan sesuai rancangan. Keduanya perlu dianalisis bersama agar metode terbaik tidak dipilih hanya berdasarkan nilai metrik tanpa mempertimbangkan kualitas dan keterwakilan simulasi.

In [13]:
evaluation_results = []
simulation_checks = []

for t_idx, target in enumerate(target_columns):

    complete_data = ground_truth[target]

    features = [f for f in meteorological_features if f != target]

    cols = list(dict.fromkeys(
        ["station", "datetime", "month", "hour", target] + features
    ))

    base = complete_data[cols].copy()
    target_std = base[target].std()

    seq_cache = build_sequence_cache(base)
    key_index = pd.MultiIndex.from_arrays(
        [base["station"].to_numpy(), base["datetime"].to_numpy()]
    )

    for g_idx, gap_name in enumerate(gap_scenarios):

        n_blocks = max_blocks_per_gap[gap_name]

        available_gaps = int((
            (actual_gap_df["target"] == target)
            & (actual_gap_df["gap"] == gap_name)
        ).sum())

        for trial, seed in enumerate(random_seeds, start=1):

            phase = "seleksi" if seed in selection_seeds else "konfirmasi"

            rng = np.random.default_rng([seed, t_idx, g_idx])

            mask, placed_lengths, placements = create_actual_gap_mask(
                base, target, actual_gap_df, gap_name,
                rng, n_blocks, seq_cache
            )

            masked_points = int(mask.sum())

            unique_source_gaps = len({(p[1], p[2], p[3]) for p in placements})

            check_base = {
                "target": target,
                "gap": gap_name,
                "trial": trial,
                "seed": seed,
                "phase": phase,
                "available_actual_gaps": available_gaps,
                "unique_source_gaps": unique_source_gaps,
                "replacement_used": bool(0 < available_gaps < n_blocks),
                "requested_blocks": n_blocks if available_gaps > 0 else 0,
                "placed_blocks": len(placed_lengths),
                "masked_points": masked_points,
                "mean_gap_length": (
                    np.mean(placed_lengths) if placed_lengths else np.nan
                ),
            }

            if masked_points == 0:
                for scenario in scenarios:
                    simulation_checks.append({
                        **check_base, "scenario": scenario,
                        "context_hidden_points": 0,
                        "mean_share_others_hidden": np.nan,
                        "share_all_others_hidden": np.nan,
                    })
                continue

            true_values = base.loc[mask, target].to_numpy()

            context_mask = build_context_mask(
                base, key_index, mask, placements, target
            )

            for scenario in scenarios:

                hidden = (mask | context_mask) if scenario == "co-missing" else mask

                artificial_data = base.copy()
                artificial_data.loc[hidden, target] = np.nan

                interp_pred = interpolate_method(artificial_data, target)
                median_pred = median_method(artificial_data, target, mask, hidden)
                nb_pred, cl_pred, ca_pred, nb_stats = neighbor_methods(
                    artificial_data, target, mask, median_pred, interp_pred
                )

                simulation_checks.append({
                    **check_base,
                    "scenario": scenario,
                    "context_hidden_points": int(hidden.sum() - mask.sum()),
                    "mean_share_others_hidden": nb_stats["mean_share_others_hidden"],
                    "share_all_others_hidden": nb_stats["share_all_others_hidden"],
                })

                methods = {
                    "Linear Interpolation": interp_pred,
                    "Conditional Median": median_pred,
                    "KNN Imputer": knn_method(
                        artificial_data, target, mask, hidden, features
                    ),
                    "Neighbor Station": nb_pred,
                    "Interp + NC (level)": cl_pred,
                    "Interp + NC (anomaly)": ca_pred,
                }

                for method_name, predictions in methods.items():

                    predicted_values = predictions.loc[mask].to_numpy()

                    if not np.isfinite(predicted_values).all():
                        raise ValueError(
                            f"{method_name} menghasilkan prediksi NaN/inf pada "
                            f"{target}, {gap_name}, trial {trial}, skenario {scenario}"
                        )

                    y_true = true_values
                    y_pred = predicted_values

                    rmse, mae, mape, bias, std_ratio = regression_metrics(
                        y_true, y_pred
                    )

                    evaluation_results.append({
                        "scenario": scenario,
                        "target": target,
                        "gap": gap_name,
                        "trial": trial,
                        "seed": seed,
                        "phase": phase,
                        "method": method_name,
                        "RMSE": rmse,
                        "nRMSE": rmse / target_std if target_std > 0 else np.nan,
                        "MAE": mae,
                        "MAPE": mape,
                        "bias": bias,
                        "nBias": bias / target_std if target_std > 0 else np.nan,
                        "std_ratio": std_ratio,
                        "evaluated_points": len(y_true),
                        "placed_blocks": len(placed_lengths),
                        "fallback_rate": (
                            nb_stats["fallback_rate"]
                            if method_name == "Neighbor Station" else np.nan
                        ),
                    })

    print(f"{target} selesai")

evaluation_df = pd.DataFrame(evaluation_results)
simulation_check_df = pd.DataFrame(simulation_checks)

PM2.5 selesai
PM10 selesai
DEWP selesai
TEMP selesai
PRES selesai
RAIN selesai
WSPM selesai


## Pemeriksaan Hasil Simulasi

Tujuan:

- Memeriksa keberhasilan proses simulasi artificial missingness, termasuk jumlah blok yang berhasil ditempatkan, jumlah titik yang disembunyikan, serta kondisi data stasiun lain.
- Mengidentifikasi kategori gap yang menggunakan sumber gap aktual secara berulang (replacement), sehingga hasil evaluasi dapat ditafsirkan dengan mempertimbangkan keterbatasan data aktual.

Yang dilakukan:

- Mengelompokkan `simulation_check_df` berdasarkan `scenario`, `target`, dan `gap`, sehingga pemeriksaan dilakukan untuk setiap kombinasi skenario, variabel, dan kategori panjang gap.
- Menghitung atau mengambil ringkasan berikut:
  - `available_actual_gaps`: jumlah gap aktual yang tersedia sebagai sumber simulasi.
  - `requested_blocks`: jumlah blok yang diminta untuk dibuat.
  - `placed_blocks_mean`: rata-rata jumlah blok yang berhasil ditempatkan.
  - `unique_source_gaps_mean`: rata-rata jumlah sumber gap aktual unik yang digunakan.
  - `replacement_used`: menunjukkan apakah pengambilan sumber gap dilakukan dengan pengembalian (replacement).
  - `masked_points_mean`: rata-rata jumlah titik data yang disembunyikan untuk evaluasi.
  - `context_hidden_mean`: rata-rata jumlah titik tambahan yang disembunyikan pada stasiun lain sebagai bagian dari skenario co-missing.
  - `mean_gap_length`: rata-rata panjang blok gap yang berhasil disimulasikan.
  - `share_others_hidden`: rata-rata proporsi data pada stasiun lain yang ikut tersembunyi.
  - `share_all_others_hidden`: proporsi simulasi ketika seluruh stasiun lain tidak memiliki nilai tersedia pada waktu yang dievaluasi.
- Menghitung `placement_rate`, yaitu perbandingan rata-rata blok yang berhasil ditempatkan dengan jumlah blok yang diminta. Nilai ini membantu menilai seberapa berhasil proses penempatan blok.
- Menampilkan ringkasan hasil simulasi menggunakan `display(check_summary.round(3))`.
- Membuat tabel `replaced` untuk menampilkan kategori pada skenario pertama (`scenarios[0]`) yang menggunakan replacement. Tabel ini memuat jumlah gap aktual yang tersedia, jumlah blok yang diminta, dan rata-rata jumlah sumber gap aktual unik yang digunakan.

Batasan:

- `placement_rate` hanya menunjukkan keberhasilan penempatan blok, bukan kualitas atau keterwakilan simulasi terhadap kondisi missing value sebenarnya.
- Penggunaan replacement memungkinkan satu gap aktual menjadi sumber bagi beberapa blok buatan. Akibatnya, jumlah blok simulasi tidak selalu mencerminkan jumlah pola missing aktual yang independen.
- `unique_source_gaps_mean` merupakan rata-rata jumlah sumber unik, sehingga tidak menunjukkan secara langsung sumber gap mana yang digunakan berulang kali.
- Nilai rata-rata dapat menyembunyikan variasi antarpercobaan (trial). Dua kategori dengan rata-rata yang sama belum tentu memiliki tingkat keberhasilan yang konsisten.
- `share_others_hidden` dan `share_all_others_hidden` membantu menggambarkan ketersediaan data stasiun lain, tetapi tidak membuktikan bahwa pola co-missing yang disimulasikan memiliki penyebab yang sama dengan kondisi sebenarnya.
- Hasil pemeriksaan bergantung pada kelengkapan dan ketepatan pencatatan di `simulation_check_df`.

Catatan: Kategori dengan `placement_rate` rendah perlu diperhatikan karena sebagian blok yang diminta tidak berhasil ditempatkan. Sementara itu, kategori dengan `replacement_used = True` dan jumlah sumber gap unik yang relatif sedikit memiliki bukti pola gap aktual yang lebih terbatas. Hasil evaluasi pada kategori tersebut sebaiknya ditafsirkan secara lebih hati-hati.

In [14]:
check_summary = (
    simulation_check_df
    .groupby(["scenario", "target", "gap"])
    .agg(
        available_actual_gaps=("available_actual_gaps", "first"),
        requested_blocks=("requested_blocks", "first"),
        placed_blocks_mean=("placed_blocks", "mean"),
        unique_source_gaps_mean=("unique_source_gaps", "mean"),
        replacement_used=("replacement_used", "first"),
        masked_points_mean=("masked_points", "mean"),
        context_hidden_mean=("context_hidden_points", "mean"),
        mean_gap_length=("mean_gap_length", "mean"),
        share_others_hidden=("mean_share_others_hidden", "mean"),
        share_all_others_hidden=("share_all_others_hidden", "mean"),
    )
    .reset_index()
)

check_summary["placement_rate"] = np.where(
    check_summary["requested_blocks"] > 0,
    check_summary["placed_blocks_mean"] / check_summary["requested_blocks"],
    np.nan
)

print("\nRingkasan keberhasilan simulasi:")
display(check_summary.round(3))

replaced = check_summary[
    (check_summary["scenario"] == scenarios[0]) & check_summary["replacement_used"]
][["target", "gap", "available_actual_gaps", "requested_blocks", "unique_source_gaps_mean"]]

print("\nKategori dengan pengambilan gap aktual berulang (replacement):")
display(replaced.round(2))


Ringkasan keberhasilan simulasi:


,scenario,target,gap,available_actual_gaps,requested_blocks,placed_blocks_mean,unique_source_gaps_mean,replacement_used,masked_points_mean,context_hidden_mean,mean_gap_length,share_others_hidden,share_all_others_hidden,placement_rate
0,co-missing,DEWP,1-3 jam,210,30,30.0,30.0,False,35.7,162.8,1.19,0.435,0.264,1.0
1,co-missing,DEWP,13-24 jam,0,0,0.0,0.0,False,0.0,0.0,NaN,NaN,NaN,NaN
2,co-missing,DEWP,25-72 jam,0,0,0.0,0.0,False,0.0,0.0,NaN,NaN,NaN,NaN
3,co-missing,DEWP,4-6 jam,26,30,30.0,17.6,True,177.9,1722.8,5.93,0.968,0.940,1.0
4,co-missing,DEWP,7-12 jam,1,20,20.0,1.0,True,140.0,0.0,7.00,0.003,0.000,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
79,independent,WSPM,13-24 jam,0,0,0.0,0.0,False,0.0,0.0,NaN,NaN,NaN,NaN
80,independent,WSPM,25-72 jam,0,0,0.0,0.0,False,0.0,0.0,NaN,NaN,NaN,NaN
81,independent,WSPM,4-6 jam,24,30,30.0,16.9,True,150.0,0.0,5.00,0.021,0.000,1.0
82,independent,WSPM,7-12 jam,0,0,0.0,0.0,False,0.0,0.0,NaN,NaN,NaN,NaN



Kategori dengan pengambilan gap aktual berulang (replacement):


,target,gap,available_actual_gaps,requested_blocks,unique_source_gaps_mean
45,DEWP,4-6 jam,26,30,17.6
46,DEWP,7-12 jam,1,20,1.0
63,PRES,4-6 jam,24,30,16.6
64,PRES,7-12 jam,1,20,1.0
69,RAIN,4-6 jam,24,30,16.9
75,TEMP,4-6 jam,26,30,18.3
76,TEMP,7-12 jam,1,20,1.0
81,WSPM,4-6 jam,24,30,16.9


Ringkasan menunjukkan bahwa seluruh blok missing value yang diminta berhasil ditempatkan dengan placement rate sebesar 100% pada kategori yang disimulasikan. Namun, beberapa kategori memerlukan pengambilan gap aktual berulang (replacement) karena jumlah gap aktual tidak mencukupi, terutama pada variabel meteorologi dengan durasi 4–6 jam dan 7–12 jam. Pengulangan paling besar terjadi pada kategori 7–12 jam yang hanya memiliki satu gap aktual. Dengan demikian, simulasi berhasil dijalankan, tetapi hasil evaluasi pada kategori yang menggunakan gap berulang perlu mempertimbangkan keterbatasan variasi sumber gap agar tidak dianggap sebagai sampel independen sepenuhnya.

## Ringkasan Evaluasi Metode Imputasi

Tujuan

Merangkum hasil evaluasi seluruh metode imputasi berdasarkan metrik kesalahan dan karakteristik prediksi untuk setiap kombinasi skenario, variabel target, kategori panjang gap, dan fase simulasi. Ringkasan ini digunakan untuk membandingkan performa metode serta mengidentifikasi metode yang memberikan hasil imputasi terbaik dan paling konsisten.

Yang Dilakukan

- Mengelompokkan hasil evaluasi berdasarkan `scenario`, `target`, `gap`, `phase`, dan `method`.
- Menghitung nilai rata-rata (`mean`) dan standar deviasi (`std`) RMSE, MAE, Bias, dan Standard Deviation Ratio untuk melihat tingkat kesalahan serta konsistensi hasil antar-trial.
- Menghitung rata-rata nRMSE, MAPE, dan nBias sebagai metrik evaluasi tambahan.
- Menghitung rata-rata jumlah titik yang dievaluasi (`evaluated_points`) untuk mengetahui banyaknya data yang digunakan dalam evaluasi.
- Menghitung rata-rata `fallback_rate` untuk memantau frekuensi penggunaan mekanisme alternatif pada metode Neighbor Station.
- Mengurutkan hasil berdasarkan skenario, variabel target, kategori gap, fase, dan nilai rata-rata RMSE dari terkecil ke terbesar.
- Membulatkan hasil hingga empat angka desimal agar lebih mudah dibaca dan dibandingkan.

Batasan

- Ringkasan hanya mencakup kombinasi yang memiliki hasil evaluasi pada `evaluation_df`.
- Nilai rata-rata dapat menyembunyikan variasi performa antar-trial sehingga perlu mempertimbangkan standar deviasi.
- Perbandingan metode harus dilakukan pada kombinasi skenario, target, kategori gap, dan fase yang sama.
- Nilai `fallback_rate` hanya relevan untuk metode Neighbor Station, sedangkan metode lain menghasilkan NaN.
- Jumlah titik evaluasi dapat berbeda antar-kombinasi sehingga perlu diperhatikan ketika membandingkan hasil.
- RMSE menjadi dasar pengurutan, tetapi pemilihan metode terbaik tetap perlu mempertimbangkan MAE, bias, variasi prediksi, dan metrik lainnya.

In [15]:
validation_summary = (
    evaluation_df
    .groupby(["scenario", "target", "gap", "phase", "method"])
    .agg(
        RMSE_mean=("RMSE", "mean"),
        RMSE_std=("RMSE", "std"),
        nRMSE_mean=("nRMSE", "mean"),
        MAE_mean=("MAE", "mean"),
        MAE_std=("MAE", "std"),
        MAPE_mean=("MAPE", "mean"),
        bias_mean=("bias", "mean"),
        bias_std=("bias", "std"),
        nBias_mean=("nBias", "mean"),
        std_ratio_mean=("std_ratio", "mean"),
        std_ratio_std=("std_ratio", "std"),
        evaluated_points=("evaluated_points", "mean"),
        fallback_rate=("fallback_rate", "mean"),
    )
    .reset_index()
    .sort_values(["scenario", "target", "gap", "phase", "RMSE_mean"])
)

print("\nRingkasan evaluasi metode imputasi:")
display(validation_summary.round(4))


Ringkasan evaluasi metode imputasi:


,scenario,target,gap,phase,method,RMSE_mean,RMSE_std,nRMSE_mean,MAE_mean,MAE_std,MAPE_mean,bias_mean,bias_std,nBias_mean,std_ratio_mean,std_ratio_std,evaluated_points,fallback_rate
4,co-missing,DEWP,1-3 jam,konfirmasi,Linear Interpolation,0.7074,0.2594,0.0513,0.5057,0.1555,8.9725,0.1188,0.2253,0.0086,0.9950,0.0137,36.8,NaN
2,co-missing,DEWP,1-3 jam,konfirmasi,Interp + NC (level),0.9150,0.2223,0.0663,0.6604,0.1393,9.9908,0.1337,0.2689,0.0097,0.9969,0.0157,36.8,NaN
1,co-missing,DEWP,1-3 jam,konfirmasi,Interp + NC (anomaly),0.9153,0.2363,0.0664,0.6579,0.1396,10.5972,0.0756,0.2606,0.0055,0.9973,0.0149,36.8,NaN
5,co-missing,DEWP,1-3 jam,konfirmasi,Neighbor Station,3.7786,0.9724,0.2739,2.5376,0.6536,61.7634,0.3849,0.2118,0.0279,0.9888,0.0931,36.8,0.2621
0,co-missing,DEWP,1-3 jam,konfirmasi,Conditional Median,6.5304,1.0193,0.4734,5.3730,0.9357,167.5751,1.1388,1.1331,0.0826,0.8436,0.1252,36.8,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
596,independent,WSPM,4-6 jam,seleksi,Interp + NC (level),0.7341,0.1186,0.5890,0.4985,0.0704,35.8173,-0.0352,0.0481,-0.0283,0.9528,0.0877,150.0,NaN
599,independent,WSPM,4-6 jam,seleksi,Neighbor Station,0.8496,0.2372,0.6817,0.5574,0.1341,39.2451,-0.0345,0.1240,-0.0277,0.8362,0.1575,150.0,0.0000
598,independent,WSPM,4-6 jam,seleksi,Linear Interpolation,0.9104,0.1646,0.7305,0.6593,0.0868,43.4769,-0.1144,0.1336,-0.0918,0.8219,0.0854,150.0,NaN
597,independent,WSPM,4-6 jam,seleksi,KNN Imputer,1.4064,0.2477,1.1285,0.9963,0.1175,62.8625,-0.1993,0.2098,-0.1599,0.6801,0.1291,150.0,NaN


Ringkasan evaluasi mencakup 600 kombinasi hasil pengujian berdasarkan skenario (co-missing dan independent), variabel target, kategori panjang gap, fase seleksi dan konfirmasi, serta metode imputasi. Evaluasi menggunakan RMSE, nRMSE, MAE, MAPE, bias, dan standard deviation ratio untuk menilai akurasi serta karakteristik hasil imputasi. Nilai rata-rata dan standar deviasi dihitung dari beberapa trial untuk melihat konsistensi performa metode. Hasil ini menjadi dasar pemilihan metode terbaik pada setiap kombinasi skenario, variabel, dan panjang gap, yang selanjutnya perlu dikonfirmasi menggunakan hasil fase konfirmasi.

## Perbandingan Berpasangan: Terbaik vs Kedua vs Referensi

Tujuan

Membandingkan performa metode imputasi secara berpasangan (paired comparison) berdasarkan nilai RMSE pada trial yang menggunakan mask missing yang identik. Analisis ini bertujuan mengetahui seberapa besar perbedaan kesalahan antara metode terbaik, metode peringkat kedua, dan metode referensi, sekaligus menilai konsistensi perbedaan tersebut antar-trial.

Yang Dilakukan

- Membentuk tabel `rmse_pivot` menggunakan `pivot_table()` agar nilai RMSE setiap metode berada dalam kolom terpisah.
- Mengelompokkan hasil berdasarkan skenario, variabel target, kategori gap, fase, dan trial sehingga perbandingan dilakukan pada kondisi simulasi yang sama.
- Membuat fungsi `paired_stats()` untuk menghitung statistik selisih RMSE berpasangan.
- Menghitung rata-rata selisih RMSE (`diff_mean`) untuk mengetahui besarnya perbedaan performa.
- Menghitung standar deviasi selisih (`diff_std`) untuk melihat variasi perbedaan antar-trial.
- Menghitung jumlah kemenangan (`wins`), yaitu jumlah trial ketika selisih RMSE bernilai positif.
- Menghitung jumlah pasangan data valid (`n`) yang digunakan dalam perbandingan.
- Menghitung interval kepercayaan 95% (`ci_low` dan `ci_high`) menggunakan distribusi t-Student apabila tersedia lebih dari satu pasangan data.

Batasan

- Perbandingan hanya dapat dilakukan pada trial yang memiliki hasil RMSE valid dari kedua metode.
- Penggunaan mask identik memastikan perbedaan RMSE dapat dibandingkan secara berpasangan pada kondisi missing yang sama.
- Jumlah trial yang relatif sedikit (5 trial) membuat estimasi rata-rata, standar deviasi, dan interval kepercayaan masih terbatas.
- Interval kepercayaan 95% digunakan sebagai informasi eksploratif, bukan sebagai dasar pengujian signifikansi formal.
- Hasil fase seleksi dan konfirmasi harus dibedakan karena fase seleksi digunakan untuk menentukan metode terbaik, sedangkan fase konfirmasi digunakan untuk mengevaluasi metode terpilih menggunakan seed berbeda.

Catatan: Interpretasi selisih RMSE bergantung pada urutan pengurangan yang digunakan ketika memanggil `paired_stats()`. Jika selisih dihitung sebagai RMSE metode pembanding dikurangi RMSE metode terbaik, nilai positif menunjukkan metode terbaik menghasilkan kesalahan lebih rendah.

In [16]:
rmse_pivot = evaluation_df.pivot_table(
    index=["scenario", "target", "gap", "phase", "trial"],
    columns="method",
    values="RMSE"
)


def paired_stats(d, prefix):
    d = d.dropna()
    n = len(d)
    out = {
        f"{prefix}_diff_mean": d.mean() if n else np.nan,
        f"{prefix}_diff_std": d.std(ddof=1) if n > 1 else np.nan,
        f"{prefix}_wins": int((d > 0).sum()),
        f"{prefix}_n": n,
        f"{prefix}_ci_low": np.nan,
        f"{prefix}_ci_high": np.nan,
    }
    if n > 1:
        half = stats.t.ppf(0.975, n - 1) * d.std(ddof=1) / np.sqrt(n)
        out[f"{prefix}_ci_low"] = d.mean() - half
        out[f"{prefix}_ci_high"] = d.mean() + half
    return out

## Penentuan Metode Terbaik dan Perbandingan pada Fase Seleksi serta Konfirmasi

Tujuan

Menentukan metode imputasi terbaik berdasarkan rata-rata RMSE pada fase seleksi, kemudian mengevaluasi konsistensi performanya pada fase konfirmasi menggunakan trial dengan random seed berbeda. Metode terpilih juga dibandingkan dengan metode peringkat kedua dan metode referensi (Linear Interpolation) untuk mengetahui keunggulan performanya pada setiap kombinasi skenario, variabel target, dan kategori gap.

Yang Dilakukan

- Memisahkan `validation_summary` menjadi fase seleksi (`seleksi`) dan konfirmasi (`konfirmasi`).
- Membuat indeks berdasarkan skenario, variabel target, kategori gap, dan metode untuk mempermudah pengambilan hasil evaluasi konfirmasi.
- Mengelompokkan hasil fase seleksi berdasarkan skenario, target, dan kategori gap.
- Memastikan setiap kombinasi memiliki hasil evaluasi pada kedua fase.
- Mengurutkan metode berdasarkan rata-rata RMSE pada fase seleksi, kemudian menetapkan metode dengan RMSE terendah sebagai metode terbaik dan metode berikutnya sebagai peringkat kedua.
- Mengambil hasil RMSE, Bias, Standard Deviation Ratio, jumlah titik evaluasi, dan fallback rate dari metode terbaik pada fase seleksi.
- Mengambil rata-rata RMSE metode terpilih pada fase konfirmasi untuk melihat performanya pada data simulasi dengan seed berbeda.
- Membandingkan metode terbaik dengan metode peringkat kedua pada fase seleksi menggunakan selisih RMSE berpasangan.
- Membandingkan metode terbaik dengan metode referensi (Linear Interpolation) pada fase seleksi dan konfirmasi menggunakan selisih RMSE berpasangan.
- Menyimpan seluruh hasil perbandingan ke dalam `rows` untuk dianalisis lebih lanjut.

Batasan

- Penentuan metode terbaik hanya didasarkan pada rata-rata RMSE fase seleksi, sehingga belum mempertimbangkan seluruh metrik evaluasi dalam proses pemeringkatan.
- Metode terbaik ditentukan secara terpisah untuk setiap kombinasi skenario, target, dan kategori gap.
- Perbandingan dengan metode kedua hanya dilakukan pada fase seleksi, sedangkan perbandingan dengan metode referensi dilakukan pada kedua fase.
- Evaluasi konfirmasi hanya menguji metode yang sudah terpilih pada fase seleksi, bukan melakukan pemilihan ulang berdasarkan hasil konfirmasi.
- Selisih RMSE berpasangan hanya dapat dihitung pada trial yang memiliki hasil valid dari kedua metode.
- Jumlah trial yang terbatas membuat hasil perbandingan dan interval kepercayaan bersifat eksploratif, bukan bukti signifikansi statistik.
- Jika metode terbaik pada fase seleksi adalah Linear Interpolation, perbandingan terhadap metode referensi tidak dilakukan karena keduanya merupakan metode yang sama.

Catatan: Hasil dari proses ini digunakan untuk melihat apakah metode yang terpilih pada fase seleksi tetap memberikan performa yang baik pada fase konfirmasi. Dengan demikian, pemilihan metode tidak hanya bergantung pada hasil terbaik selama seleksi, tetapi juga mempertimbangkan kestabilan performanya ketika diuji menggunakan seed yang berbeda.

In [17]:
sel_summary = validation_summary[validation_summary["phase"] == "seleksi"]
conf_summary = validation_summary[validation_summary["phase"] == "konfirmasi"]
conf_idx = conf_summary.set_index(["scenario", "target", "gap", "method"])

rows = []

for (sc, tg, gp), grp in sel_summary.groupby(["scenario", "target", "gap"]):

    for ph in ("seleksi", "konfirmasi"):
        if (sc, tg, gp, ph) not in rmse_pivot.index:
            raise ValueError(f"Tidak ada trial fase '{ph}' untuk {sc}, {tg}, {gp}.")

    grp = grp.sort_values("RMSE_mean")
    best = grp.iloc[0]
    best_m = best["method"]

    sel_trials = rmse_pivot.loc[(sc, tg, gp, "seleksi")]
    conf_trials = rmse_pivot.loc[(sc, tg, gp, "konfirmasi")]

    row = {
        "scenario": sc, "target": tg, "gap": gp,
        "best_method": best_m,
        "sel_RMSE_mean": best["RMSE_mean"],
        "sel_RMSE_std": best["RMSE_std"],
        "best_sel_bias_mean": best["bias_mean"],
        "best_sel_std_ratio_mean": best["std_ratio_mean"],
        "evaluated_points": best["evaluated_points"],
        "fallback_rate": best["fallback_rate"],
        "conf_RMSE_mean": conf_idx.loc[(sc, tg, gp, best_m), "RMSE_mean"],
        "second_method": np.nan,
    }

    if len(grp) > 1:
        second = grp.iloc[1]["method"]
        row["second_method"] = second
        row.update(paired_stats(sel_trials[second] - sel_trials[best_m], "second"))

    if best_m != reference_method and reference_method in sel_trials:
        row.update(paired_stats(
            sel_trials[reference_method] - sel_trials[best_m], "sel_vs_ref"
        ))
        row.update(paired_stats(
            conf_trials[reference_method] - conf_trials[best_m], "vs_ref"
        ))

    rows.append(row)

## Menentukan Metode Imputasi yang Direkomendasikan

Tujuan

Menentukan metode imputasi yang paling layak direkomendasikan untuk setiap kombinasi skenario, variabel target, dan panjang gap. Pemilihan tidak hanya didasarkan pada nilai RMSE terendah saat seleksi, tetapi juga mempertimbangkan konsistensi keunggulan metode dan hasil pengujian pada data konfirmasi dibandingkan interpolasi linear sebagai metode referensi.

Yang Dilakukan

- Membuat DataFrame `comparison` dari hasil perbandingan metode pada tahap sebelumnya.
- Menghitung relative gain, yaitu rata-rata selisih RMSE dibandingkan dengan RMSE metode terbaik, untuk melihat besarnya keunggulan secara relatif.
- Mengklasifikasikan hasil perbandingan berdasarkan dua kriteria: konsistensi kemenangan pada seluruh trial dan besarnya peningkatan relatif terhadap batas minimum yang ditentukan.
- Membandingkan metode terbaik dengan metode terbaik kedua pada trial seleksi, serta dengan interpolasi linear pada trial seleksi dan konfirmasi.
- Menentukan metode rekomendasi menggunakan aturan konservatif: metode selain interpolasi linear hanya dipilih apabila menunjukkan keunggulan yang konsisten dan memenuhi batas peningkatan relatif dibandingkan interpolasi linear pada trial konfirmasi.
- Menggabungkan metrik evaluasi pada trial konfirmasi milik metode yang direkomendasikan, meliputi RMSE, nRMSE, MAE, bias, dan rasio standar deviasi.

Batasan

- Penilaian konsistensi didasarkan pada jumlah trial yang terbatas, sehingga kemenangan pada seluruh trial bukan merupakan bukti signifikansi statistik.
- Nilai relative gain menggunakan RMSE rata-rata sebagai dasar perbandingan. Hasil klasifikasi bergantung pada nilai `min_relative_gain` dan `min_trials_consistent` yang ditetapkan.
- Perbandingan dengan metode terbaik kedua dilakukan pada trial seleksi sehingga bersifat deskriptif dan belum bebas dari selection bias.
- Perbandingan dengan interpolasi linear pada trial konfirmasi lebih relevan untuk menilai kemampuan generalisasi metode terpilih karena menggunakan trial yang berbeda dari proses seleksi.
- Interval kepercayaan 95% tetap tersedia sebagai informasi pendukung, tetapi tidak digunakan sebagai kriteria penentuan metode karena jumlah trial terbatas dan variasi panjang blok missing dapat memengaruhi estimasinya.

In [18]:
comparison = pd.DataFrame(rows)

for prefix in ("second", "sel_vs_ref", "vs_ref"):
    for c in (f"{prefix}_n", f"{prefix}_wins", f"{prefix}_diff_mean"):
        if c not in comparison:
            comparison[c] = np.nan

comparison["second_rel_gain"] = comparison["second_diff_mean"] / comparison["sel_RMSE_mean"]
comparison["sel_vs_ref_rel_gain"] = comparison["sel_vs_ref_diff_mean"] / comparison["sel_RMSE_mean"]
comparison["vs_ref_rel_gain"] = comparison["vs_ref_diff_mean"] / comparison["conf_RMSE_mean"]


def classify(prefix, label_ok, label_small, label_no):
    n = comparison[f"{prefix}_n"]
    consistent = (n >= min_trials_consistent) & (comparison[f"{prefix}_wins"] == n)
    meaningful = comparison[f"{prefix}_rel_gain"] >= min_relative_gain
    return np.select(
        [consistent & meaningful, consistent & ~meaningful],
        [label_ok, label_small],
        default=label_no
    )


comparison["verdict_vs_second"] = classify(
    "second", "unggul konsisten", "konsisten tapi selisih kecil", "relatif berdekatan"
)

is_ref = comparison["best_method"] == reference_method
comparison["verdict_vs_ref"] = np.where(
    is_ref,
    "(sendiri adalah referensi)",
    classify("vs_ref", "lebih baik dari interpolasi",
             "lebih baik tapi selisih kecil", "tidak terbukti lebih baik dari interpolasi")
)

comparison["recommended_method"] = np.where(
    is_ref | (comparison["verdict_vs_ref"] == "lebih baik dari interpolasi"),
    comparison["best_method"],
    reference_method
)

rec_metrics = (
    conf_summary[[
        "scenario", "target", "gap", "method",
        "RMSE_mean", "nRMSE_mean", "MAE_mean", "bias_mean", "std_ratio_mean",
    ]]
    .rename(columns={
        "method": "recommended_method",
        "RMSE_mean": "rec_RMSE_mean",
        "nRMSE_mean": "rec_nRMSE_mean",
        "MAE_mean": "rec_MAE_mean",
        "bias_mean": "rec_bias_mean",
        "std_ratio_mean": "rec_std_ratio_mean",
    })
)

comparison = comparison.merge(
    rec_metrics,
    on=["scenario", "target", "gap", "recommended_method"],
    how="left"
)

if comparison["rec_RMSE_mean"].isna().any():
    raise ValueError("Metrik metode rekomendasi tidak ditemukan untuk sebagian sel.")

## Penyusunan Keputusan Metode Imputasi Berdasarkan Hasil Evaluasi

Tujuan

Tahapan ini bertujuan menyusun rekomendasi metode imputasi untuk setiap kombinasi skenario, variabel, dan kategori panjang gap berdasarkan hasil evaluasi simulasi. Rekomendasi tidak hanya mempertimbangkan tingkat kesalahan imputasi, tetapi juga konsistensi hasil evaluasi dan ketersediaan gap aktual sebagai dasar penerapan metode pada data asli.

Yang Dilakukan

- Menggabungkan informasi dukungan data dari hasil pemeriksaan simulasi (`check_summary`) ke tabel perbandingan metode (`comparison`), meliputi jumlah gap aktual yang tersedia, jumlah blok yang diminta dan berhasil ditempatkan, jumlah gap sumber unik, serta penggunaan gap pengganti (replacement).
- Mengklasifikasikan dukungan data menjadi tiga kategori, yaitu lemah, terbatas, dan memadai berdasarkan jumlah gap aktual yang tersedia.
- Menyusun hasil seleksi metode dengan menampilkan metode terbaik dan metode terbaik kedua berdasarkan rata-rata RMSE, beserta perbandingan kinerja keduanya.
- Menyajikan hasil konfirmasi berdasarkan evaluasi terhadap metode referensi (reference method), termasuk selisih relatif RMSE, jumlah kemenangan (wins), dan hasil keputusan (verdict).
- Menetapkan metode yang direkomendasikan dengan mempertimbangkan hasil konfirmasi serta menampilkan metrik pendukung, seperti RMSE, normalized RMSE, bias, dan rasio standar deviasi.
- Menyusun tabel keputusan akhir untuk mempermudah interpretasi dan menentukan metode yang layak diterapkan pada setiap kombinasi skenario, variabel, dan kategori gap.

Batasan

- Hasil seleksi dan konfirmasi berlaku untuk skenario simulasi serta konfigurasi evaluasi yang telah ditetapkan, sehingga tidak secara otomatis menjamin kinerja yang sama pada data asli.
- Kategori dukungan data hanya menggambarkan ketersediaan gap aktual untuk simulasi, bukan kualitas atau keterwakilan seluruh pola missing value pada dataset.
- Penggunaan replacement menunjukkan bahwa sebagian blok simulasi memerlukan gap pengganti karena keterbatasan gap aktual. Oleh karena itu, hasil evaluasi pada kategori dengan dukungan lemah perlu diinterpretasikan secara lebih hati-hati.
- RMSE, bias, dan rasio standar deviasi merupakan metrik evaluasi yang saling melengkapi. Metode dengan RMSE terendah belum tentu paling baik dalam mempertahankan karakteristik distribusi data.
- Tabel ini merupakan dasar pengambilan keputusan metode imputasi, bukan hasil penerapan maupun validasi akhir pada data asli.

In [19]:
support_info = check_summary[[
    "scenario", "target", "gap",
    "available_actual_gaps", "requested_blocks", "placed_blocks_mean",
    "unique_source_gaps_mean", "replacement_used",
]]
comparison = comparison.merge(support_info, on=["scenario", "target", "gap"], how="left")

comparison["support"] = np.select(
    [
        comparison["available_actual_gaps"] < weak_support_below,
        comparison["available_actual_gaps"] < adequate_support_from,
    ],
    ["lemah", "terbatas"],
    default="memadai"
)

decision_cols = [
    "scenario", "target", "gap",
    "best_method", "sel_RMSE_mean",
    "second_method", "second_rel_gain", "second_wins", "second_n", "verdict_vs_second",
    "sel_vs_ref_rel_gain", "sel_vs_ref_wins", "sel_vs_ref_n",
    "conf_RMSE_mean",
    "vs_ref_rel_gain", "vs_ref_wins", "vs_ref_n", "verdict_vs_ref",
    "recommended_method",
    "rec_RMSE_mean", "rec_nRMSE_mean", "rec_bias_mean", "rec_std_ratio_mean",
    "available_actual_gaps", "unique_source_gaps_mean", "replacement_used",
    "placed_blocks_mean", "support",
]

print("\nKeputusan per skenario, variabel, dan gap:")
display(comparison[decision_cols].round(4))


Keputusan per skenario, variabel, dan gap:


,scenario,target,gap,best_method,sel_RMSE_mean,second_method,second_rel_gain,second_wins,second_n,verdict_vs_second,...,recommended_method,rec_RMSE_mean,rec_nRMSE_mean,rec_bias_mean,rec_std_ratio_mean,available_actual_gaps,unique_source_gaps_mean,replacement_used,placed_blocks_mean,support
0,co-missing,DEWP,1-3 jam,Linear Interpolation,0.9039,Interp + NC (level),0.1869,4,5,relatif berdekatan,...,Linear Interpolation,0.7074,0.0513,0.1188,0.9950,210,30.0,False,30.0,memadai
1,co-missing,DEWP,4-6 jam,Linear Interpolation,1.6340,Interp + NC (anomaly),0.0037,4,5,relatif berdekatan,...,Linear Interpolation,1.5202,0.1102,0.3445,0.9759,26,17.6,True,30.0,terbatas
2,co-missing,DEWP,7-12 jam,Interp + NC (anomaly),0.9553,Interp + NC (level),0.0060,3,5,relatif berdekatan,...,Linear Interpolation,1.3123,0.0951,0.1291,0.9983,1,1.0,True,20.0,lemah
3,co-missing,PM10,1-3 jam,Linear Interpolation,22.6238,Interp + NC (anomaly),0.0602,2,5,relatif berdekatan,...,Linear Interpolation,21.1947,0.2310,1.1973,0.9443,2045,30.0,False,30.0,memadai
4,co-missing,PM10,13-24 jam,Interp + NC (anomaly),46.9518,Interp + NC (level),0.0007,3,5,relatif berdekatan,...,Interp + NC (anomaly),45.3020,0.4937,1.5232,1.0240,62,15.0,False,15.0,memadai
5,co-missing,PM10,25-72 jam,Neighbor Station,25.3481,Interp + NC (anomaly),0.0429,3,5,relatif berdekatan,...,Neighbor Station,35.3925,0.3857,-2.0620,0.9495,21,6.0,False,6.0,terbatas
6,co-missing,PM10,4-6 jam,Interp + NC (level),30.2230,Interp + NC (anomaly),0.0025,5,5,konsisten tapi selisih kecil,...,Linear Interpolation,37.4002,0.4075,2.7740,0.9637,141,30.0,False,30.0,memadai
7,co-missing,PM10,7-12 jam,Interp + NC (anomaly),33.1789,Interp + NC (level),0.0013,1,5,relatif berdekatan,...,Interp + NC (anomaly),29.3926,0.3203,1.8527,1.0016,48,20.0,False,20.0,memadai
8,co-missing,PM10,>72 jam,Neighbor Station,37.7935,Interp + NC (level),0.1326,5,5,unggul konsisten,...,Neighbor Station,40.4354,0.4406,7.7723,0.9600,4,3.0,False,3.0,lemah
9,co-missing,PM2.5,1-3 jam,Interp + NC (anomaly),14.7737,Linear Interpolation,0.0054,2,5,relatif berdekatan,...,Linear Interpolation,17.1933,0.2127,0.4561,1.0019,2474,30.0,False,30.0,memadai


Tabel keputusan merangkum pemilihan metode imputasi terbaik untuk setiap kombinasi skenario, variabel target, dan kategori panjang gap. Keputusan mempertimbangkan hasil fase seleksi dan konfirmasi, perbandingan performa metode terbaik dengan metode kedua, serta ketersediaan gap aktual dan penggunaan replacement. Kolom `recommended_method` menunjukkan metode yang direkomendasikan setelah mempertimbangkan hasil konfirmasi dan kekuatan data pendukung (`support`). Hasil menunjukkan bahwa metode terbaik tidak selalu sama dengan metode yang direkomendasikan, sehingga pemilihan akhir tidak hanya didasarkan pada nilai RMSE terendah, tetapi juga konsistensi dan keterbatasan data simulasi.

## Penyajian Ringkasan Rekomendasi Metode Imputasi Berdasarkan Variabel dan Kategori Gap

Tujuan

Menyajikan hasil rekomendasi metode imputasi dalam bentuk tabel ringkas berdasarkan variabel target dan kategori panjang gap pada setiap skenario. Tahapan ini bertujuan mempermudah identifikasi metode imputasi yang direkomendasikan untuk masing-masing variabel dan panjang gap, serta melihat pola perbedaan rekomendasi antar skenario.

Yang Dilakukan

- Menentukan urutan kategori gap berdasarkan konfigurasi `gap_scenarios` agar penyajian tabel konsisten dengan urutan kategori yang digunakan dalam simulasi.
- Melakukan pemfilteran data `comparison` berdasarkan masing-masing skenario.
- Mengubah hasil rekomendasi ke dalam format pivot dengan `target` sebagai indeks (baris), `gap` sebagai kolom, dan `recommended_method` sebagai nilai.
- Menyesuaikan urutan kolom berdasarkan kategori gap yang tersedia agar sesuai dengan urutan yang telah ditentukan.
- Menampilkan tabel rekomendasi metode imputasi secara terpisah untuk setiap skenario.

Batasan

- Tabel hanya menampilkan nama metode yang direkomendasikan tanpa menyertakan nilai metrik evaluasi, seperti RMSE, bias, dan rasio standar deviasi.
- Informasi tingkat dukungan data simulasi tidak ditampilkan, sehingga hasil perlu diinterpretasikan bersama tabel keputusan utama.
- Rekomendasi yang ditampilkan merupakan hasil dari proses seleksi dan konfirmasi sebelumnya, bukan hasil pemilihan metode baru.
- Kombinasi variabel dan kategori gap yang tidak memiliki hasil rekomendasi akan ditampilkan sebagai nilai kosong (`NaN`).
- Perbandingan antar skenario hanya menunjukkan perbedaan metode yang direkomendasikan, bukan secara langsung menunjukkan skenario dengan performa imputasi terbaik.

Catatan: Tabel ini digunakan sebagai ringkasan akhir untuk melihat distribusi rekomendasi metode imputasi berdasarkan variabel target dan panjang gap pada setiap skenario. Dasar pemilihan metode serta tingkat dukungan data tetap mengacu pada tabel keputusan sebelumnya.

In [20]:
gap_order = list(gap_scenarios)

for sc in scenarios:
    pivot = (
        comparison[comparison["scenario"] == sc]
        .pivot(index="target", columns="gap", values="recommended_method")
    )
    pivot = pivot.reindex(columns=[g for g in gap_order if g in pivot.columns])
    print(f"\nMetode yang direkomendasikan - skenario {sc}:")
    display(pivot)


Metode yang direkomendasikan - skenario independent:


gap,1-3 jam,4-6 jam,7-12 jam,13-24 jam,25-72 jam,>72 jam
target,,,,,,
DEWP,Linear Interpolation,Interp + NC (anomaly),Linear Interpolation,NaN,NaN,NaN
PM10,Linear Interpolation,Interp + NC (level),Interp + NC (anomaly),Interp + NC (anomaly),Neighbor Station,Neighbor Station
PM2.5,Linear Interpolation,Interp + NC (level),Interp + NC (anomaly),Interp + NC (level),Interp + NC (anomaly),Neighbor Station
PRES,Linear Interpolation,Linear Interpolation,Linear Interpolation,NaN,NaN,NaN
RAIN,Linear Interpolation,Linear Interpolation,NaN,NaN,NaN,NaN
TEMP,Linear Interpolation,Interp + NC (level),Interp + NC (level),NaN,NaN,NaN
WSPM,Linear Interpolation,Interp + NC (anomaly),NaN,NaN,NaN,NaN



Metode yang direkomendasikan - skenario co-missing:


gap,1-3 jam,4-6 jam,7-12 jam,13-24 jam,25-72 jam,>72 jam
target,,,,,,
DEWP,Linear Interpolation,Linear Interpolation,Linear Interpolation,NaN,NaN,NaN
PM10,Linear Interpolation,Linear Interpolation,Interp + NC (anomaly),Interp + NC (anomaly),Neighbor Station,Neighbor Station
PM2.5,Linear Interpolation,Interp + NC (anomaly),Linear Interpolation,Interp + NC (level),Interp + NC (anomaly),Neighbor Station
PRES,Linear Interpolation,Linear Interpolation,Linear Interpolation,NaN,NaN,NaN
RAIN,Linear Interpolation,Linear Interpolation,NaN,NaN,NaN,NaN
TEMP,Linear Interpolation,Linear Interpolation,Interp + NC (level),NaN,NaN,NaN
WSPM,Linear Interpolation,Linear Interpolation,NaN,NaN,NaN,NaN


Berdasarkan hasil evaluasi, rekomendasi metode imputasi menunjukkan perbedaan antara skenario independent dan co-missing, terutama pada gap 4–24 jam. Pada skenario independent, penggunaan Neighbor Correction (NC) lebih banyak direkomendasikan untuk beberapa variabel, sedangkan pada skenario co-missing Linear Interpolation masih direkomendasikan pada beberapa kondisi gap yang sama. Untuk gap 1–3 jam, Linear Interpolation menjadi metode yang paling konsisten direkomendasikan pada kedua skenario. Sementara itu, pada gap 25–72 jam hingga lebih dari 72 jam, rekomendasi semakin selektif dan didominasi oleh Neighbor Station untuk `PM10` dan `PM2.5`, sedangkan variabel lainnya lebih banyak dipertahankan sebagai NaN. Perbedaan ini menunjukkan bahwa pemilihan metode imputasi tidak hanya bergantung pada panjang gap, tetapi juga pada ketersediaan informasi pendukung ketika missing value terjadi secara bersamaan pada beberapa variabel.

## Analisis Dampak Co-Missing terhadap Performa Metode Imputasi Berbasis Stasiun Lain

Tujuan

Menganalisis pengaruh kondisi co-missing terhadap performa metode imputasi yang memanfaatkan informasi dari stasiun pengamatan lain. Analisis ini bertujuan mengetahui apakah hilangnya data secara bersamaan pada beberapa stasiun menyebabkan perubahan tingkat kesalahan imputasi dibandingkan kondisi independent, ketika missing value diasumsikan terjadi secara independen.

Yang Dilakukan

- Memastikan skenario `independent` dan `co-missing` tersedia sebelum melakukan analisis perbandingan.
- Mengelompokkan `evaluation_df` berdasarkan skenario, variabel target, kategori gap, dan metode imputasi.
- Menghitung rata-rata RMSE dari setiap kombinasi kelompok untuk memperoleh ringkasan performa metode pada masing-masing skenario.
- Mengubah hasil pengelompokan ke dalam format pivot agar nilai RMSE pada skenario `independent` dan `co-missing` dapat dibandingkan secara berdampingan.
- Menghitung perubahan absolut RMSE (delta RMSE) dengan mengurangkan RMSE skenario `independent` dari RMSE skenario `co-missing`.
- Menghitung perubahan relatif RMSE dalam persentase terhadap RMSE skenario `independent`.
- Membatasi hasil analisis pada variabel PM2.5 dan PM10, kemudian menampilkan perbandingan untuk setiap kategori gap dan metode imputasi.

Batasan

- Analisis hanya dilakukan apabila kedua skenario, yaitu `independent` dan `co-missing`, tersedia dalam hasil evaluasi.
- Perbandingan menggunakan rata-rata RMSE dari seluruh hasil evaluasi yang tersedia, bukan perbandingan berpasangan berdasarkan random seed yang sama.
- Perubahan RMSE menunjukkan perbedaan performa antara kedua skenario, tetapi tidak secara langsung membuktikan bahwa co-missing merupakan satu-satunya penyebab perubahan tersebut.
- Analisis hanya berfokus pada PM2.5 dan PM10, sehingga hasilnya tidak dapat digeneralisasikan ke seluruh variabel dalam dataset.
- Perubahan relatif RMSE bergantung pada nilai RMSE skenario `independent`. Nilai dasar yang sangat kecil dapat menghasilkan persentase perubahan yang besar.
- Analisis ini mengevaluasi perubahan tingkat kesalahan imputasi, bukan secara langsung mengukur kontribusi atau ketergantungan informasi dari masing-masing stasiun pengamatan.

Catatan: Nilai delta RMSE positif menunjukkan bahwa RMSE meningkat pada skenario co-missing, sedangkan nilai negatif menunjukkan bahwa RMSE menurun. Dengan demikian, hasil analisis dapat digunakan untuk mengidentifikasi metode imputasi yang performanya paling terpengaruh ketika informasi dari stasiun lain berpotensi ikut mengalami kehilangan data secara bersamaan.

In [21]:
if set(["independent", "co-missing"]).issubset(set(scenarios)):

    pooled_rmse = (
        evaluation_df
        .groupby(["scenario", "target", "gap", "method"], as_index=False)["RMSE"]
        .mean()
    )

    effect = pooled_rmse.pivot_table(
        index=["target", "gap", "method"],
        columns="scenario",
        values="RMSE"
    )
    effect["delta (co-missing - independent)"] = (
        effect["co-missing"] - effect["independent"]
    )
    effect["delta_%"] = effect["delta (co-missing - independent)"] / effect["independent"] * 100

    print("\nPerubahan RMSE akibat co-missing (PM2.5 dan PM10):")
    display(
        effect.reset_index()
        .query("target in ['PM2.5', 'PM10']")
        .round(3)
    )


Perubahan RMSE akibat co-missing (PM2.5 dan PM10):


scenario,target,gap,method,co-missing,independent,delta (co-missing - independent),delta_%
18,PM10,1-3 jam,Conditional Median,102.736,102.637,0.099,0.096
19,PM10,1-3 jam,Interp + NC (anomaly),22.874,21.773,1.102,5.060
20,PM10,1-3 jam,Interp + NC (level),23.159,21.796,1.363,6.254
21,PM10,1-3 jam,KNN Imputer,91.965,96.094,-4.129,-4.297
22,PM10,1-3 jam,Linear Interpolation,21.909,21.909,0.000,0.000
...,...,...,...,...,...,...,...
85,PM2.5,>72 jam,Interp + NC (anomaly),36.526,35.676,0.850,2.382
86,PM2.5,>72 jam,Interp + NC (level),36.548,35.696,0.852,2.388
87,PM2.5,>72 jam,KNN Imputer,75.248,73.531,1.717,2.335
88,PM2.5,>72 jam,Linear Interpolation,85.044,85.044,0.000,0.000


Berdasarkan perbandingan RMSE antara skenario co-missing dan independent, pengaruh co-missing terhadap akurasi imputasi `PM2.5` dan `PM10` bervariasi bergantung pada metode dan panjang gap. Sebagian besar metode menunjukkan peningkatan RMSE pada skenario co-missing, yang mengindikasikan penurunan akurasi imputasi ketika missing value terjadi bersamaan pada variabel lain. Namun, terdapat beberapa pengecualian, seperti KNN Imputer pada `PM10` dengan gap 1–3 jam yang justru mengalami penurunan RMSE sebesar 4,297%. Linear Interpolation tidak menunjukkan perubahan RMSE karena metode ini hanya memanfaatkan nilai sebelum dan sesudah gap pada variabel target. Secara umum, hasil ini menunjukkan bahwa co-missing dapat memengaruhi performa metode imputasi, terutama metode yang memanfaatkan informasi dari variabel lain atau stasiun tetangga, meskipun besar pengaruhnya tidak selalu sama pada setiap kondisi.

## Diagnostik Perbandingan Co-Missing Aktual dan Simulasi

Tujuan

Membandingkan tingkat co-missing yang ditemukan pada data asli dengan tingkat co-missing yang terbentuk dalam skenario simulasi. Analisis ini bertujuan mengevaluasi sejauh mana pola kehilangan data secara bersamaan pada stasiun lain dalam simulasi mendekati kondisi aktual, khususnya pada periode gap yang dikelompokkan berdasarkan kategori panjangnya.

Yang Dilakukan

- Mengambil informasi gap aktual dari `actual_gap_df` untuk setiap variabel target dan kategori gap.
- Melakukan iterasi terhadap setiap gap aktual berdasarkan waktu mulai, waktu berakhir, dan stasiun yang mengalami missing value.
- Mengambil template data stasiun lain menggunakan fungsi `get_template()` untuk mengidentifikasi kondisi missing value pada stasiun selain stasiun yang sedang dievaluasi.
- Menghitung proporsi jam ketika terdapat setidaknya satu stasiun lain yang mengalami missing value melalui `actual_share_others_missing`.
- Menghitung proporsi jam ketika seluruh stasiun lain mengalami missing value secara bersamaan melalui `actual_share_all_others_missing`.
- Menggabungkan hasil diagnostik aktual dengan informasi co-missing simulasi dari `check_summary` pada skenario yang ditentukan melalui `scenarios[-1]`.
- Membandingkan proporsi co-missing aktual dan simulasi berdasarkan variabel target dan kategori gap.
- Menampilkan hasil perbandingan dalam satu tabel agar perbedaan pola co-missing aktual dan simulasi dapat diamati.

Batasan

- Analisis aktual hanya dilakukan pada periode gap yang teridentifikasi dalam `actual_gap_df`, sehingga tidak menggambarkan seluruh periode pengamatan.
- Perhitungan menggunakan stasiun lain sebagai pembanding dan tidak memasukkan stasiun yang sedang mengalami gap sebagai bagian dari kelompok stasiun lain.
- Proporsi others missing menunjukkan kondisi ketika minimal satu stasiun lain mengalami missing value, sedangkan all others missing menunjukkan kondisi ketika seluruh stasiun lain mengalami missing value pada jam yang sama.
- Hasil simulasi dapat sedikit lebih tinggi dibandingkan kondisi aktual karena cakupan stasiun yang digunakan dalam simulasi dapat mencakup stasiun yang tidak tersedia pada complete-case.
- Perbandingan ini hanya mengevaluasi kesesuaian proporsi co-missing, bukan kesamaan waktu kejadian, durasi, atau pola temporal kehilangan data secara keseluruhan.
- Kedekatan nilai proporsi aktual dan simulasi tidak secara otomatis membuktikan bahwa simulasi telah merepresentasikan seluruh karakteristik co-missing pada data asli.

Catatan: Diagnostik ini digunakan untuk menilai kewajaran pola co-missing yang dibentuk dalam simulasi terhadap kondisi data asli. Jika proporsi simulasi jauh lebih tinggi atau lebih rendah daripada aktual, perbedaan tersebut perlu dipertimbangkan ketika menginterpretasikan hasil evaluasi metode imputasi pada skenario co-missing.

In [22]:
actual_rows = []

for t in target_columns:
    sub = actual_gap_df[(actual_gap_df["target"] == t)]
    for gp, g in sub.groupby("gap"):
        hours = 0
        share_sum = 0.0
        all_sum = 0
        for st, et, station in zip(g["start_time"], g["end_time"], g["station"]):
            cols, tmpl = get_template(t, st, et)
            others = [i for i, c in enumerate(cols) if c != station]
            block = tmpl[:, others]
            hours += block.shape[0]
            share_sum += block.mean(axis=1).sum()
            all_sum += int(block.all(axis=1).sum())
        actual_rows.append({
            "target": t, "gap": gp,
            "actual_share_others_missing": share_sum / hours,
            "actual_share_all_others_missing": all_sum / hours,
        })

actual_comissing = pd.DataFrame(actual_rows)

sim_cm = check_summary[check_summary["scenario"] == scenarios[-1]][
    ["target", "gap", "share_others_hidden", "share_all_others_hidden"]
]

diag = actual_comissing.merge(sim_cm, on=["target", "gap"], how="left")

print(f"\nCo-missing aktual vs simulasi (skenario {scenarios[-1]}):")
display(diag.round(3))


Co-missing aktual vs simulasi (skenario co-missing):


,target,gap,actual_share_others_missing,actual_share_all_others_missing,share_others_hidden,share_all_others_hidden
0,PM2.5,1-3 jam,0.191,0.113,0.210,0.124
1,PM2.5,13-24 jam,0.155,0.127,0.189,0.143
2,PM2.5,25-72 jam,0.029,0.000,0.053,0.000
3,PM2.5,4-6 jam,0.366,0.333,0.372,0.322
4,PM2.5,7-12 jam,0.245,0.209,0.270,0.217
5,PM2.5,>72 jam,0.024,0.005,0.039,0.007
6,PM10,1-3 jam,0.235,0.155,0.239,0.159
7,PM10,13-24 jam,0.203,0.179,0.226,0.188
8,PM10,25-72 jam,0.032,0.000,0.038,0.000
9,PM10,4-6 jam,0.457,0.418,0.461,0.416


Berdasarkan perbandingan co-missing aktual dan simulasi, proporsi missing value pada variabel lain dalam simulasi secara umum cukup mendekati kondisi aktual, meskipun terdapat beberapa perbedaan pada kombinasi target dan panjang gap tertentu. Pada `PM2.5` dan `PM10`, selisih proporsi missing value relatif kecil pada sebagian besar kategori gap, misalnya `PM10` dengan gap 1–3 jam memiliki proporsi missing pada variabel lain sebesar 23,5% pada kondisi aktual dan 23,9% pada simulasi. Sementara itu, pada variabel meteorologi seperti `DEWP`, `TEMP`, `PRES`, dan `RAIN`, proporsi missing value pada gap 4–6 jam sangat tinggi, yaitu sekitar 96,5–100% pada kondisi aktual dan 96,8–99,7% pada simulasi. Pola ini menunjukkan bahwa simulasi secara umum berhasil merepresentasikan tingkat co-missing aktual, meskipun belum sepenuhnya identik. Dengan demikian, hasil evaluasi pada skenario co-missing dapat digunakan untuk menggambarkan performa metode imputasi dalam kondisi missing value yang melibatkan beberapa variabel secara bersamaan, dengan tetap mempertimbangkan perbedaan distribusi antara kondisi aktual dan simulasi.

## Penyusunan Tabel Aturan Penanganan Missing Value Berdasarkan Hasil Evaluasi

Tujuan

Menyusun aturan penanganan missing value untuk setiap variabel target dan kategori panjang gap berdasarkan hasil evaluasi metode imputasi pada skenario terakhir yang dijalankan. Tahapan ini bertujuan menentukan metode imputasi yang dapat digunakan, membedakan keputusan yang telah didukung hasil evaluasi dengan keputusan yang masih memerlukan pengujian tambahan, serta mencatat tingkat ketersediaan data pendukung secara terpisah.

Yang Dilakukan

- Menentukan skenario terakhir dalam `scenarios` sebagai dasar penyusunan aturan penanganan missing value, sehingga menggunakan skenario `co-missing` apabila skenario tersebut dijalankan terakhir.
- Mengambil hasil evaluasi dari `comparison` berdasarkan skenario yang telah ditentukan.
- Menetapkan status keputusan (`status_keputusan`) berdasarkan hasil konfirmasi dan perbandingan terhadap metode referensi (Linear Interpolation), dengan ketentuan:
  - `didukung`: metode terbaik merupakan metode referensi atau metode yang direkomendasikan bukan metode referensi.
  - `batas`: metode referensi tetap digunakan sementara karena metode kompleks menunjukkan keunggulan relatif minimal sebesar `min_relative_gain` dan menang pada seluruh trial kecuali satu trial, sehingga masih memerlukan tambahan pengujian.
- Mengubah nama kolom `support` menjadi `dukungan_data` untuk memperjelas informasi tingkat ketersediaan gap aktual.
- Mengambil informasi metode yang direkomendasikan, status keputusan, dukungan data, jumlah gap aktual, jumlah gap sumber unik, penggunaan replacement, serta metrik evaluasi metode yang direkomendasikan pada fase konfirmasi.
- Menyertakan informasi metode terbaik pada fase seleksi dan hasil perbandingannya terhadap metode referensi sebagai dasar pendukung keputusan.
- Membentuk kombinasi lengkap antara seluruh variabel target dan kategori gap menggunakan `MultiIndex.from_product()`.
- Menggabungkan kombinasi lengkap tersebut dengan hasil evaluasi agar kategori gap yang tidak memiliki gap aktual tetap tercatat secara eksplisit.
- Memberikan keterangan `tidak dievaluasi` pada metode yang tidak memiliki hasil evaluasi dan `tidak ada gap aktual` pada status keputusannya.
- Menampilkan tabel aturan penanganan missing value yang mencakup rekomendasi metode, status keputusan, dukungan data, dan metrik evaluasi pendukung.

Batasan

- Penyusunan aturan hanya menggunakan hasil evaluasi dari skenario terakhir dalam `scenarios`, sehingga keputusan tidak secara langsung mewakili hasil seluruh skenario.
- Status keputusan didasarkan pada hasil konfirmasi dan perbandingan terhadap metode referensi, bukan pada pengujian signifikansi statistik.
- Kategori `didukung` menunjukkan bahwa metode yang direkomendasikan memenuhi logika keputusan dalam kode, tetapi tidak secara otomatis membuktikan bahwa metode tersebut unggul secara statistik atau telah tervalidasi pada seluruh kondisi data asli.
- Kategori `batas` menunjukkan bahwa keunggulan metode kompleks belum memenuhi seluruh kriteria keputusan, sehingga interpolasi masih digunakan sementara dan membutuhkan tambahan trial.
- Tingkat `dukungan_data` dilaporkan secara terpisah dari `status_keputusan`. Dengan demikian, metode yang berstatus `didukung` belum tentu memiliki dukungan data yang memadai.
- Metrik `rec_*` merupakan hasil evaluasi metode yang direkomendasikan pada fase konfirmasi, bukan otomatis metrik dari `best_method` pada fase seleksi.
- Kategori yang tidak memiliki gap aktual tidak dapat dievaluasi melalui simulasi berbasis gap aktual, sehingga tidak diberikan rekomendasi metode imputasi.
- Jumlah gap sumber unik dan penggunaan replacement tetap perlu diperhatikan karena dapat memengaruhi keterwakilan pola gap dalam simulasi.

Catatan: Tabel ini merupakan hasil sintesis dari tahapan evaluasi sebelumnya dan digunakan sebagai dasar penyusunan aturan penanganan missing value pada data asli. Keputusan penggunaan metode perlu mempertimbangkan tiga aspek secara bersamaan, yaitu metode yang direkomendasikan, status keputusan berdasarkan hasil konfirmasi, dan tingkat dukungan data aktual. Status `didukung` tidak boleh diartikan sebagai jaminan bahwa seluruh kondisi penerapan metode sudah tervalidasi.

In [23]:
final_scenario = scenarios[-1]
rule_src = comparison[comparison["scenario"] == final_scenario].copy()


def rule_status(row):
    if row["best_method"] == reference_method or row["recommended_method"] != reference_method:
        return "didukung"
    n = row["vs_ref_n"]
    if (row["vs_ref_rel_gain"] >= min_relative_gain) and (row["vs_ref_wins"] >= n - 1):
        return "batas"
    return "didukung"


rule_src["status_keputusan"] = rule_src.apply(rule_status, axis=1)
rule_src = rule_src.rename(columns={"support": "dukungan_data"})

rule_table = rule_src[[
    "target", "gap",
    "recommended_method", "status_keputusan", "dukungan_data",
    "available_actual_gaps", "unique_source_gaps_mean", "replacement_used",
    "rec_RMSE_mean", "rec_nRMSE_mean", "rec_MAE_mean",
    "rec_bias_mean", "rec_std_ratio_mean",
    "best_method", "vs_ref_rel_gain", "vs_ref_wins", "vs_ref_n",
]]

full_grid = pd.MultiIndex.from_product(
    [target_columns, list(gap_scenarios)], names=["target", "gap"]
).to_frame(index=False)

rule_table = full_grid.merge(rule_table, on=["target", "gap"], how="left")
not_eval = rule_table["recommended_method"].isna()
rule_table.loc[not_eval, "recommended_method"] = "tidak dievaluasi"
rule_table.loc[not_eval, "status_keputusan"] = "tidak ada gap aktual"
rule_table.loc[not_eval, "dukungan_data"] = "-"

print(f"\nAturan penanganan missing value (dasar: skenario {final_scenario}):")
display(rule_table.round(4))


Aturan penanganan missing value (dasar: skenario co-missing):


,target,gap,recommended_method,status_keputusan,dukungan_data,available_actual_gaps,unique_source_gaps_mean,replacement_used,rec_RMSE_mean,rec_nRMSE_mean,rec_MAE_mean,rec_bias_mean,rec_std_ratio_mean,best_method,vs_ref_rel_gain,vs_ref_wins,vs_ref_n
0,PM2.5,1-3 jam,Linear Interpolation,didukung,memadai,2474.0,30.0,False,17.1933,0.2127,9.8821,0.4561,1.0019,Interp + NC (anomaly),-0.0239,2.0,5.0
1,PM2.5,4-6 jam,Interp + NC (anomaly),didukung,memadai,180.0,30.0,False,18.3396,0.2269,10.8105,0.4861,1.0006,Interp + NC (anomaly),0.1211,5.0,5.0
2,PM2.5,7-12 jam,Linear Interpolation,batas,memadai,61.0,20.0,False,24.2302,0.2997,15.1734,0.2197,0.9082,Interp + NC (anomaly),0.1920,4.0,5.0
3,PM2.5,13-24 jam,Interp + NC (level),didukung,memadai,85.0,15.0,False,41.6470,0.5152,21.1518,6.6379,1.0038,Interp + NC (level),0.3497,5.0,5.0
4,PM2.5,25-72 jam,Interp + NC (anomaly),didukung,memadai,31.0,6.0,False,30.1164,0.3725,19.6909,0.4500,0.9485,Interp + NC (anomaly),0.9952,5.0,5.0
5,PM2.5,>72 jam,Neighbor Station,didukung,terbatas,6.0,3.0,False,28.5737,0.3535,17.4057,-2.7433,0.8715,Neighbor Station,2.0326,5.0,5.0
6,PM10,1-3 jam,Linear Interpolation,didukung,memadai,2045.0,30.0,False,21.1947,0.2310,11.0854,1.1973,0.9443,Linear Interpolation,NaN,NaN,NaN
7,PM10,4-6 jam,Linear Interpolation,batas,memadai,141.0,30.0,False,37.4002,0.4075,22.6041,2.7740,0.9637,Interp + NC (level),0.0286,4.0,5.0
8,PM10,7-12 jam,Interp + NC (anomaly),didukung,memadai,48.0,20.0,False,29.3926,0.3203,20.2598,1.8527,1.0016,Interp + NC (anomaly),0.2407,5.0,5.0
9,PM10,13-24 jam,Interp + NC (anomaly),didukung,memadai,62.0,15.0,False,45.3020,0.4937,29.8021,1.5232,1.0240,Interp + NC (anomaly),0.1527,5.0,5.0


Berdasarkan hasil evaluasi pada skenario co-missing, aturan penanganan missing value ditetapkan dengan mempertimbangkan performa metode imputasi, ketersediaan gap aktual, jumlah gap sumber yang unik, serta konsistensi hasil antar-trial. Linear Interpolation direkomendasikan untuk sebagian besar gap pendek, terutama 1–3 jam, sedangkan kombinasi interpolasi dan Neighbor Correction (NC) lebih banyak digunakan pada PM2.5 dan PM10 untuk gap menengah hingga panjang. Neighbor Station dipilih untuk gap panjang pada kedua variabel tersebut, meskipun dukungan datanya masih terbatas hingga lemah karena jumlah gap aktual dan sumber gap yang tersedia relatif sedikit. Sementara itu, variabel meteorologi seperti `DEWP`, `TEMP`, `PRES`, `RAIN`, dan `WSPM` umumnya hanya direkomendasikan untuk gap pendek hingga 4–6 jam. Beberapa kondisi gap yang memiliki sampel sangat terbatas tetap diberi status batas atau dukungan lemah, sehingga hasilnya perlu ditafsirkan secara hati-hati. Adapun kondisi tanpa gap aktual tidak dievaluasi dan dipertahankan sebagai NaN. Dengan demikian, aturan akhir tidak hanya didasarkan pada metode dengan RMSE terendah, tetapi juga mempertimbangkan kecukupan data dan keandalan hasil evaluasi.

## Penyajian Ringkasan Aturan Penanganan Missing Value Berdasarkan Variabel dan Kategori Gap

Tujuan

Menyajikan aturan penanganan missing value dalam bentuk tabel ringkas berdasarkan variabel target dan kategori panjang gap. Tahapan ini bertujuan mempermudah identifikasi metode imputasi yang direkomendasikan sekaligus mempertahankan informasi penting mengenai status keputusan dan tingkat dukungan data pada setiap kombinasi variabel dan gap.

Yang Dilakukan

- Membuat pemetaan keterangan tambahan berdasarkan `status_keputusan`, dengan memberikan penanda `(batas)` pada keputusan yang masih memerlukan pengujian tambahan.
- Membuat fungsi `support_mark()` untuk menambahkan informasi dukungan data pada metode yang memiliki dukungan terbatas atau lemah.
- Menampilkan jumlah gap aktual dalam keterangan tambahan apabila tingkat dukungan data belum memadai, dengan format `[data kategori: jumlah gap]`.
- Menggabungkan nama metode yang direkomendasikan, status keputusan, dan informasi dukungan data ke dalam kolom `label`.
- Mengubah tabel aturan menjadi format pivot dengan `target` sebagai baris, `gap` sebagai kolom, dan `label` sebagai nilai.
- Menyesuaikan urutan baris berdasarkan `target_columns` dan urutan kolom berdasarkan `gap_scenarios`.
- Menampilkan ringkasan aturan penanganan missing value untuk seluruh kombinasi variabel dan kategori gap.

Batasan

- Tabel hanya menampilkan nama metode dan keterangan status serta dukungan data, tanpa menyertakan metrik evaluasi secara numerik.
- Informasi dukungan data hanya ditambahkan untuk kategori terbatas dan lemah, sedangkan kategori memadai tidak diberikan penanda tambahan.
- Status `(batas)` menunjukkan bahwa metode interpolasi masih digunakan sementara dan memerlukan tambahan trial, bukan berarti metode tersebut telah terbukti optimal.
- Ringkasan tetap bergantung pada hasil keputusan di tahap sebelumnya dan tidak melakukan evaluasi atau pemilihan metode ulang.
- Kombinasi yang tidak memiliki gap aktual tetap ditampilkan berdasarkan keterangan `tidak dievaluasi`, tanpa rekomendasi metode imputasi.

Catatan: Tabel ini merupakan representasi ringkas dari aturan penanganan missing value yang telah disusun pada tahap sebelumnya. Keterangan tambahan pada setiap sel membantu membedakan rekomendasi yang memiliki dukungan data memadai dengan rekomendasi yang masih memiliki keterbatasan data atau memerlukan pengujian lanjutan.

In [24]:
status_mark = {"didukung": "", "batas": " (batas)", "tidak ada gap aktual": ""}


def support_mark(row):
    if row["dukungan_data"] in ("memadai", "-"):
        return ""
    return f" [data {row['dukungan_data']}: {int(row['available_actual_gaps'])} gap]"


rule_table["label"] = (
    rule_table["recommended_method"]
    + rule_table["status_keputusan"].map(status_mark)
    + rule_table.apply(support_mark, axis=1)
)

rule_pivot = rule_table.pivot(index="target", columns="gap", values="label")
rule_pivot = rule_pivot.reindex(index=target_columns, columns=list(gap_scenarios))

print("\nRingkasan aturan (variabel x gap):")
display(rule_pivot)


Ringkasan aturan (variabel x gap):


gap,1-3 jam,4-6 jam,7-12 jam,13-24 jam,25-72 jam,>72 jam
target,,,,,,
PM2.5,Linear Interpolation,Interp + NC (anomaly),Linear Interpolation (batas),Interp + NC (level),Interp + NC (anomaly),Neighbor Station [data terbatas: 6 gap]
PM10,Linear Interpolation,Linear Interpolation (batas),Interp + NC (anomaly),Interp + NC (anomaly),Neighbor Station [data terbatas: 21 gap],Neighbor Station [data lemah: 4 gap]
DEWP,Linear Interpolation,Linear Interpolation [data terbatas: 26 gap],Linear Interpolation (batas) [data lemah: 1 gap],tidak dievaluasi,tidak dievaluasi,tidak dievaluasi
TEMP,Linear Interpolation,Linear Interpolation [data terbatas: 26 gap],Interp + NC (level) [data lemah: 1 gap],tidak dievaluasi,tidak dievaluasi,tidak dievaluasi
PRES,Linear Interpolation,Linear Interpolation [data terbatas: 24 gap],Linear Interpolation [data lemah: 1 gap],tidak dievaluasi,tidak dievaluasi,tidak dievaluasi
RAIN,Linear Interpolation,Linear Interpolation [data terbatas: 24 gap],tidak dievaluasi,tidak dievaluasi,tidak dievaluasi,tidak dievaluasi
WSPM,Linear Interpolation,Linear Interpolation [data terbatas: 24 gap],tidak dievaluasi,tidak dievaluasi,tidak dievaluasi,tidak dievaluasi


Ringkasan aturan penanganan missing value menunjukkan bahwa pemilihan metode bergantung pada variabel dan panjang gap. Linear Interpolation menjadi metode utama untuk gap 1–3 jam pada seluruh variabel, sedangkan pada gap 4–6 jam mulai digunakan kombinasi interpolasi dan Neighbor Correction (NC), khususnya untuk `PM2.5`. Untuk gap 7–24 jam, `PM2.5` dan `PM10` lebih banyak menggunakan kombinasi interpolasi dan NC, sementara variabel meteorologi memiliki rekomendasi yang lebih terbatas. Pada gap 25–72 jam hingga lebih dari 72 jam, imputasi hanya direkomendasikan untuk `PM2.5` dan `PM10` dengan metode kombinasi interpolasi dan NC atau Neighbor Station. Beberapa rekomendasi memiliki keterbatasan dukungan data, terutama pada gap panjang dan kondisi dengan jumlah gap aktual yang sangat sedikit. Sementara itu, kategori tanpa gap aktual tidak dievaluasi sehingga tidak ditetapkan metode imputasi. Secara keseluruhan, aturan ini menunjukkan pendekatan imputasi yang selektif dengan mempertimbangkan karakteristik variabel, panjang gap, dan kecukupan data evaluasi.